In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Global ocean-side wetland lifeline weakening analysis (FigureV1, robust version).

This script performs a fully standalone ocean-side analysis for coastal wetlands.
It intentionally does NOT produce wetland-side exposure results.

The primary manuscript analysis excludes locations north of 66 degrees N.
A full-domain calculation is retained only as a sensitivity benchmark, allowing
the effect of the Arctic exclusion to be quantified without changing the
original physical definitions or distance-kernel formulation.

Main outputs
------------
1. Figure1_ocean_lifeline_weakening_main.jpg
2. Figure2_ocean_lifeline_severity_main.jpg
3. Sensitivity figures for scenario/year, wave/quantile, and kernel choices
4. Tables and main-case GeoTIFF rasters

Important robustness fixes
--------------------------
1. Avoids NUMEXPR thread conflicts by respecting NUMEXPR_MAX_THREADS.
2. Fixes PROJ/GDAL conflicts on SCC by:
   - setting PROJ_DATA/PROJ_LIB from pyproj before importing rasterio;
   - writing geographic output rasters using a literal WGS84 WKT string,
     rather than relying on EPSG:4326 database lookup.
3. Caches wetland aggregation as both NPZ and GeoTIFF, so a failed GeoTIFF write
   will not force recomputing the expensive wetland aggregation from scratch.

All comments are in English as requested.
"""

from __future__ import annotations

# =============================================================================
# Environment setup must happen before importing rasterio/pandas heavily
# =============================================================================

import os
import sys
from pathlib import Path

MAX_CORES = 24
MAX_MEMORY_GB = 120

_existing_numexpr_max = os.environ.get("NUMEXPR_MAX_THREADS", "")
try:
    _numexpr_max = int(_existing_numexpr_max) if _existing_numexpr_max.strip() else MAX_CORES
except Exception:
    _numexpr_max = MAX_CORES
SAFE_NUMEXPR_THREADS = max(1, min(MAX_CORES, _numexpr_max))

os.environ["NUMEXPR_NUM_THREADS"] = str(SAFE_NUMEXPR_THREADS)
os.environ.setdefault("NUMEXPR_MAX_THREADS", str(SAFE_NUMEXPR_THREADS))

for _name in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "GDAL_NUM_THREADS",
):
    os.environ[_name] = str(MAX_CORES)

os.environ.setdefault("GDAL_CACHEMAX", str(int(MAX_MEMORY_GB * 1024 * 0.08)))

PROJ_DATA_DIR = None
try:
    import pyproj  # type: ignore
    _proj_dir = pyproj.datadir.get_data_dir()
    if _proj_dir and os.path.exists(_proj_dir):
        PROJ_DATA_DIR = str(_proj_dir)
        os.environ["PROJ_DATA"] = PROJ_DATA_DIR
        os.environ["PROJ_LIB"] = PROJ_DATA_DIR
except Exception:
    PROJ_DATA_DIR = None

WGS84_WKT = (
    'GEOGCS["WGS 84",'
    'DATUM["WGS_1984",'
    'SPHEROID["WGS 84",6378137,298.257223563]],'
    'PRIMEM["Greenwich",0],'
    'UNIT["degree",0.0174532925199433],'
    'AXIS["Longitude",EAST],'
    'AXIS["Latitude",NORTH]]'
)

# =============================================================================
# Imports
# =============================================================================

import gc
import json
import math
import logging
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

import rasterio
from rasterio.enums import Resampling
from rasterio.transform import Affine
from rasterio.warp import reproject
from rasterio.windows import Window

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix

try:
    from tqdm import tqdm
except Exception:
    def tqdm(x: Iterable, **kwargs):
        return x

try:
    import cartopy.crs as ccrs  # type: ignore
    import cartopy.feature as cfeature  # type: ignore
    HAS_CARTOPY = True
except Exception:
    HAS_CARTOPY = False

# =============================================================================
# User-adjustable paths
# =============================================================================

WETLAND_PATH = Path(
    "/projectnb/nasa-marsh/Hangjie/Louisiana/cedric/global/mask/wetland/"
    "GWL_FCS30D_2022_coastal_wetland_global_1_2_3.tif"
)

FINAL_MASK_PATH = Path(
    "/projectnb/nasa-marsh/Hangjie/Louisiana/cedric/global/mask/final_mask.tif"
)

SHEAR_DIR = Path(
    "/projectnb/nasa-marsh/Hangjie/Louisiana/cedric/global/shear_change"
)

OUTPUT_ROOT = Path(
    "/projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures"
)
DATA_DIR = OUTPUT_ROOT / "data"

RUN_LABEL = "WSCIagg_wavegrid_McCowan"

# =============================================================================
# Analysis controls
# =============================================================================

COARSE_RES_DEG = 0.10
COARSE_LON_MIN = -180.0
COARSE_LON_MAX = 180.0
COARSE_LAT_MIN = -90.0
COARSE_LAT_MAX = 90.0

# Primary manuscript domain. Only northern latitudes above 66N are excluded;
# Southern Hemisphere points are left unchanged.
MAX_ANALYSIS_LAT = 66.0

REBUILD_WETLAND_AGGREGATION = False
REBUILD_CONNECTION_TABLE = False

SELECTED_VERSIONS = ["withVLM"]
SELECTED_SCENARIOS = ["ssp126", "ssp245", "ssp585"]
SELECTED_YEARS = [2050, 2100]
SELECTED_CANONICAL_QUANTILES = ["q017", "q050", "q083"]

ANALYSIS_WAVE_CASES = [
    "combined_mean",
    "windwave_mean",
    "combined_p90",
    "windwave_p90",
]

MAIN_WAVE_CASE = "combined_mean"
MAIN_VERSION = "withVLM"
MAIN_SCENARIO = "ssp585"
MAIN_YEAR = 2100
MAIN_QUANTILE = "q050"
MAIN_KERNEL_LABEL = "lambda50_Dmax150"

KERNEL_SPECS = [
    {"lambda_km": 25.0, "dmax_km": 100.0, "label": "lambda25_Dmax100"},
    {"lambda_km": 50.0, "dmax_km": 150.0, "label": "lambda50_Dmax150"},
    {"lambda_km": 100.0, "dmax_km": 300.0, "label": "lambda100_Dmax300"},
]

WETLAND_AGGREGATION_WINDOW_SIZE = 4096
DISTANCE_CHUNK_SIZE = 12000

# =============================================================================
# Figure controls
# =============================================================================

FIG_DPI = 520
JPG_QUALITY = 96

FONT_FAMILY = "serif"
BASE_FONT = 9.2
SMALL_FONT = 8.0
TINY_FONT = 7.0
PANEL_FONT = 12.0

FIG1_SIZE = (10.0, 10.0)
FIG1_SECOND_ROW_LEFT = 0.075
FIG1_SECOND_ROW_RIGHT = 0.925
FIG1_SECOND_ROW_WIDTH = FIG1_SECOND_ROW_RIGHT - FIG1_SECOND_ROW_LEFT
FIG1_SECOND_ROW_BOTTOM = 0.080
FIG1_SECOND_ROW_HEIGHT = 0.275
FIG1_SECOND_ROW_GAP = 0.055
FIG1_PANEL_B_LEFT = FIG1_SECOND_ROW_LEFT
FIG1_PANEL_B_WIDTH = (FIG1_SECOND_ROW_WIDTH - FIG1_SECOND_ROW_GAP) / 2.0
FIG1_PANEL_C_LEFT = FIG1_PANEL_B_LEFT + FIG1_PANEL_B_WIDTH + FIG1_SECOND_ROW_GAP
FIG1_PANEL_C_WIDTH = FIG1_PANEL_B_WIDTH
FIG1_PANEL_A_LEFT = FIG1_SECOND_ROW_LEFT
FIG1_PANEL_A_RIGHT = FIG1_SECOND_ROW_RIGHT
FIG1_PANEL_A_WIDTH = FIG1_PANEL_A_RIGHT - FIG1_PANEL_A_LEFT
FIG1_PANEL_A_BOTTOM = 0.445
FIG1_PANEL_A_HEIGHT = 0.435
FIG1_CBAR_WIDTH = 0.43
FIG1_CBAR_LEFT = FIG1_PANEL_A_LEFT + 0.5 * (FIG1_PANEL_A_WIDTH - FIG1_CBAR_WIDTH)
FIG1_CBAR_BOTTOM = 0.410
FIG1_CBAR_HEIGHT = 0.020

FIG2_SIZE = (12.0, 8.8)
HEATMAP_FIGSIZE = (9.3, 6.3)
KERNEL_FIGSIZE = (9.2, 4.8)

MAP_SIZE_LEVEL_AREAS = (2.0, 10.0, 28.0)
MAP_SIZE_BIN_EDGE_QUANTILES = (45.0, 82.0)
MAP_SIZE_LEGEND_QUANTILES = (25.0, 62.0, 92.0)
MAP_ALPHA = 0.92

VIOLIN_WIDTH = 0.36
VIOLIN_BINS = 220
VIOLIN_DENSE_POINTS = 700
VIOLIN_SMOOTH_SIGMA_BINS = 3.2
VIOLIN_RANGE_QUANTILES = (0.003, 0.997)
VIOLIN_TAIL_PAD_FRACTION = 0.10
VIOLIN_EDGE_COLOR = "0.18"
VIOLIN_EDGE_WIDTH = 0.55
VIOLIN_ALPHA = 0.96
SUMMARY_LINE_COLOR = "0.05"
SUMMARY_WHISKER_COLOR = "0.30"

PCT_VMIN = -30.0
PCT_VMAX = 0.0
PCT_TICKS = [-30, -20, -10, 0]

WETLAND_STYLE = {
    1: {"name": "mangrove", "pretty": "Mangrove", "color": "#2b8c61"},
    2: {"name": "salt_marsh", "pretty": "Salt marsh", "color": "#c58c2b"},
    3: {"name": "tidal_flat", "pretty": "Tidal flat", "color": "#4c78a8"},
}

TABLE_DIR = DATA_DIR / "tables"
FIGURE_DIR = OUTPUT_ROOT
RASTER_DIR = DATA_DIR / "rasters"
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
LOG_DIR = DATA_DIR / "logs"
METADATA_DIR = DATA_DIR / "metadata"

ARCTIC_WETLAND_SUMMARY_CSV = DATA_DIR / "Arctic_exclusion_wetland_summary.csv"
MANUSCRIPT_SUMMARY_CSV = DATA_DIR / "Arctic_exclusion_summary_for_manuscript.csv"
ARCTIC_WETLAND_FIG = FIGURE_DIR / "FigureS_Arctic_exclusion_wetland_impacts.jpg"

for _p in [TABLE_DIR, FIGURE_DIR, RASTER_DIR, INTERMEDIATE_DIR, LOG_DIR, METADATA_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(LOG_DIR / "FigureV1_ocean_side_analysis_robust.log", mode="w"),
    ],
)

logging.info("SAFE_NUMEXPR_THREADS = %s", SAFE_NUMEXPR_THREADS)
logging.info("PROJ_DATA_DIR = %s", PROJ_DATA_DIR)

plt.rcParams.update(
    {
        "font.family": FONT_FAMILY,
        "font.size": BASE_FONT,
        "axes.labelsize": BASE_FONT,
        "xtick.labelsize": SMALL_FONT,
        "ytick.labelsize": SMALL_FONT,
        "legend.fontsize": SMALL_FONT,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

EARTH_RADIUS_KM = 6371.0088
NODATA_FLOAT = -9999.0

@dataclass(frozen=True)
class AnalysisCase:
    case_id: str
    wave_case: str
    version: str
    scenario: str
    year: int
    quantile_raw: str
    quantile_canonical: str
    band: int
    path: Path

@dataclass
class WetlandPoints:
    lon: np.ndarray
    lat: np.ndarray
    row: np.ndarray
    col: np.ndarray
    linear_index: np.ndarray
    area_by_class: Dict[int, np.ndarray]
    total_area: np.ndarray

@dataclass
class OffshorePoints:
    lon: np.ndarray
    lat: np.ndarray
    row: np.ndarray
    col: np.ndarray
    linear_index: np.ndarray
    case_values: np.ndarray
    case_ids: List[str]
    grid: Dict[str, object]

@dataclass
class KernelLoadResult:
    label: str
    lambda_km: float
    dmax_km: float
    total_load: np.ndarray
    class_loads: Dict[int, np.ndarray]

def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)

def save_json(path: Path, obj: object) -> None:
    ensure_dir(path.parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def save_jpg(fig: plt.Figure, path: Path, dpi: int = FIG_DPI) -> None:
    ensure_dir(path.parent)
    fig.savefig(path, dpi=dpi, format="jpg", pil_kwargs={"quality": JPG_QUALITY})
    plt.close(fig)
    logging.info("Saved figure: %s", path)

def normalize_lon_180(lon: np.ndarray) -> np.ndarray:
    return ((np.asarray(lon, dtype=float) + 180.0) % 360.0) - 180.0

def case_key(wave_case: str, version: str, scenario: str, year: int, qcanon: str) -> str:
    return f"{wave_case}__{version}__{scenario}__{int(year)}__{qcanon}"

def main_case_id() -> str:
    return case_key(MAIN_WAVE_CASE, MAIN_VERSION, MAIN_SCENARIO, MAIN_YEAR, MAIN_QUANTILE)


def latitude_domain_masks(
    wetland_points: WetlandPoints,
    offshore_points: OffshorePoints,
    max_lat: Optional[float],
) -> Tuple[np.ndarray, np.ndarray]:
    """Return wetland/offshore masks for a northern latitude cutoff."""
    if max_lat is None:
        wet_mask = np.ones(len(wetland_points.lat), dtype=bool)
        off_mask = np.ones(len(offshore_points.lat), dtype=bool)
    else:
        wet_mask = np.isfinite(wetland_points.lat) & (wetland_points.lat <= float(max_lat))
        off_mask = np.isfinite(offshore_points.lat) & (offshore_points.lat <= float(max_lat))
    return wet_mask, off_mask


def parse_case_id(case_id: str) -> Dict[str, object]:
    parts = str(case_id).split("__")
    if len(parts) != 5:
        return {"case_id": case_id}
    return {
        "wave_case": parts[0],
        "version": parts[1],
        "scenario": parts[2],
        "year": int(parts[3]),
        "quantile_canonical": parts[4],
    }

def scenario_label(s: str) -> str:
    mapping = {
        "ssp119": "SSP1-1.9",
        "ssp126": "SSP1-2.6",
        "ssp245": "SSP2-4.5",
        "ssp370": "SSP3-7.0",
        "ssp585": "SSP5-8.5",
    }
    return mapping.get(str(s).lower(), str(s).upper())

def quantile_label_pretty(q: str) -> str:
    return {"q017": "P17", "q050": "Median", "q083": "P83"}.get(str(q).lower(), str(q))

def wave_case_label(w: str) -> str:
    mapping = {
        "combined_mean": "Combined mean",
        "combined_p90": "Combined p90",
        "windwave_mean": "Wind-wave mean",
        "windwave_p90": "Wind-wave p90",
    }
    return mapping.get(w, w)

def canonical_quantile_label(label: str) -> str:
    s = str(label).strip().lower().replace("-", "").replace("_", "").replace(" ", "")
    mapping = {
        "q017": "q017", "p17": "q017", "17": "q017", "0.17": "q017",
        "q050": "q050", "p50": "q050", "median": "q050", "50": "q050", "0.50": "q050", "0.5": "q050",
        "q083": "q083", "p83": "q083", "83": "q083", "0.83": "q083",
    }
    return mapping.get(s, str(label))

def lonlat_to_xyz(lon_deg: np.ndarray, lat_deg: np.ndarray) -> np.ndarray:
    lon = np.deg2rad(np.asarray(lon_deg, dtype=np.float64))
    lat = np.deg2rad(np.asarray(lat_deg, dtype=np.float64))
    coslat = np.cos(lat)
    return np.column_stack([coslat * np.cos(lon), coslat * np.sin(lon), np.sin(lat)]).astype(np.float64)

def haversine_km(lon1_deg, lat1_deg, lon2_deg, lat2_deg) -> np.ndarray:
    lon1 = np.deg2rad(np.asarray(lon1_deg, dtype=np.float64))
    lat1 = np.deg2rad(np.asarray(lat1_deg, dtype=np.float64))
    lon2 = np.deg2rad(np.asarray(lon2_deg, dtype=np.float64))
    lat2 = np.deg2rad(np.asarray(lat2_deg, dtype=np.float64))
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))

def safe_nanmax(arr: np.ndarray, fallback: float = 1.0) -> float:
    arr = np.asarray(arr, dtype=float)
    ok = np.isfinite(arr)
    if ok.sum() == 0:
        return fallback
    return float(np.nanmax(arr[ok]))

def weighted_quantile(values: np.ndarray, weights: np.ndarray, quantiles: Sequence[float]) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    q = np.asarray(quantiles, dtype=np.float64)
    ok = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if ok.sum() == 0:
        return np.full(len(q), np.nan, dtype=np.float64)
    v = values[ok]
    w = weights[ok]
    order = np.argsort(v)
    v = v[order]
    w = w[order]
    cw = np.cumsum(w)
    total = cw[-1]
    if total <= 0:
        return np.full(len(q), np.nan, dtype=np.float64)
    cdf = (cw - 0.5 * w) / total
    return np.interp(q, cdf, v, left=v[0], right=v[-1])

def weighted_summary(values: np.ndarray, weights: np.ndarray) -> Dict[str, float]:
    ok = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if ok.sum() == 0:
        return {k: np.nan for k in ["n", "weight_sum", "mean", "p05", "p25", "median", "p75", "p95", "min", "max"]}
    v = values[ok]
    w = weights[ok]
    q05, q25, q50, q75, q95 = weighted_quantile(v, w, [0.05, 0.25, 0.50, 0.75, 0.95])
    return {
        "n": int(ok.sum()),
        "weight_sum": float(np.sum(w)),
        "mean": float(np.average(v, weights=w)),
        "p05": float(q05),
        "p25": float(q25),
        "median": float(q50),
        "p75": float(q75),
        "p95": float(q95),
        "min": float(np.nanmin(v)),
        "max": float(np.nanmax(v)),
    }

def add_fraction_metrics(row: Dict[str, object], values: np.ndarray, weights: np.ndarray) -> None:
    ok = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if ok.sum() == 0:
        row["frac_linked_to_any_decrease"] = np.nan
        row["frac_linked_to_gt10_decrease"] = np.nan
        row["frac_linked_to_gt25_decrease"] = np.nan
        row["frac_linked_to_gt50_decrease"] = np.nan
        return
    v = values[ok]
    w = weights[ok]
    total = float(np.sum(w))
    row["frac_linked_to_any_decrease"] = float(np.sum(w[v < 0]) / total)
    row["frac_linked_to_gt10_decrease"] = float(np.sum(w[v <= -10]) / total)
    row["frac_linked_to_gt25_decrease"] = float(np.sum(w[v <= -25]) / total)
    row["frac_linked_to_gt50_decrease"] = float(np.sum(w[v <= -50]) / total)

def top_fraction_share(values: np.ndarray, fractions: Sequence[float] = (0.01, 0.05, 0.10)) -> Dict[str, float]:
    x = np.asarray(values, dtype=np.float64)
    x = x[np.isfinite(x) & (x > 0)]
    out: Dict[str, float] = {}
    if x.size == 0:
        for f in fractions:
            out[f"top_{int(round(f * 100))}pct_share"] = np.nan
        return out
    x_sorted = np.sort(x)[::-1]
    total = float(np.sum(x_sorted))
    for f in fractions:
        n = max(1, int(math.ceil(f * len(x_sorted))))
        out[f"top_{int(round(f * 100))}pct_share"] = float(np.sum(x_sorted[:n]) / total)
    return out

def pct_tif_path(wave_case: str) -> Path:
    return SHEAR_DIR / f"delta_tau_percent_{wave_case}_{RUN_LABEL}_AR6_relative_to_2025_all_scenarios.tif"

def qa_tif_path() -> Path:
    return SHEAR_DIR / f"inputs_QA_{RUN_LABEL}.tif"

def crs_to_safe_wkt(crs_obj) -> str:
    if crs_obj is None:
        return WGS84_WKT
    try:
        wkt = crs_obj.to_wkt()
        if wkt:
            return wkt
    except Exception:
        pass
    s = str(crs_obj)
    if s.upper().startswith("EPSG:"):
        return WGS84_WKT
    if "GEOGCS" in s or "GEODCRS" in s or "PROJCS" in s:
        return s
    return WGS84_WKT

def transforms_match(a: Affine, b: Affine, tol: float = 1e-10) -> bool:
    return all(abs(x - y) <= tol for x, y in zip(a, b))

def raster_grid(path: Path) -> Dict[str, object]:
    with rasterio.open(path) as src:
        transform = src.transform
        bounds = src.bounds
        width = src.width
        height = src.height
        crs_wkt = crs_to_safe_wkt(src.crs)
        nodata = src.nodata
        lon = bounds.left + (np.arange(width, dtype=np.float64) + 0.5) * transform.a
        lat = bounds.top + (np.arange(height, dtype=np.float64) + 0.5) * transform.e
    return {
        "transform": transform,
        "bounds": bounds,
        "width": width,
        "height": height,
        "crs": crs_wkt,
        "nodata": nodata,
        "lon": lon,
        "lat": lat,
        "xres": transform.a,
        "yres": abs(transform.e),
    }

def coarse_grid() -> Dict[str, object]:
    width = int(round((COARSE_LON_MAX - COARSE_LON_MIN) / COARSE_RES_DEG))
    height = int(round((COARSE_LAT_MAX - COARSE_LAT_MIN) / COARSE_RES_DEG))
    transform = Affine(COARSE_RES_DEG, 0.0, COARSE_LON_MIN, 0.0, -COARSE_RES_DEG, COARSE_LAT_MAX)
    lon = COARSE_LON_MIN + (np.arange(width, dtype=np.float64) + 0.5) * COARSE_RES_DEG
    lat = COARSE_LAT_MAX - (np.arange(height, dtype=np.float64) + 0.5) * COARSE_RES_DEG
    return {
        "transform": transform,
        "bounds": rasterio.coords.BoundingBox(COARSE_LON_MIN, COARSE_LAT_MIN, COARSE_LON_MAX, COARSE_LAT_MAX),
        "width": width,
        "height": height,
        "crs": WGS84_WKT,
        "nodata": NODATA_FLOAT,
        "lon": lon,
        "lat": lat,
        "xres": COARSE_RES_DEG,
        "yres": COARSE_RES_DEG,
    }

def read_band(path: Path, band: int) -> np.ndarray:
    with rasterio.open(path) as src:
        arr = src.read(band).astype(np.float32)
        nodata = src.nodata
    if nodata is not None:
        arr[np.isclose(arr, nodata)] = np.nan
    arr[~np.isfinite(arr)] = np.nan
    return arr

def read_band_metadata(path: Path) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    with rasterio.open(path) as src:
        for band in range(1, src.count + 1):
            tags = {str(k): str(v) for k, v in src.tags(band).items()}
            rows.append(
                {
                    "band": band,
                    "description": src.descriptions[band - 1] if src.descriptions else "",
                    "metric": tags.get("metric", ""),
                    "units": tags.get("units", ""),
                    "wave_case": tags.get("wave_case", ""),
                    "slr_version": tags.get("slr_version", ""),
                    "slr_scenario": tags.get("slr_scenario", ""),
                    "slr_target_year": pd.to_numeric(tags.get("slr_target_year", np.nan), errors="coerce"),
                    "slr_quantile_label": tags.get("slr_quantile_label", ""),
                }
            )
    return pd.DataFrame(rows)

def find_qa_band(path: Path, text: str) -> int:
    target = text.lower()
    with rasterio.open(path) as src:
        for band, desc in enumerate(src.descriptions, start=1):
            if desc and target in desc.lower():
                return band
    raise KeyError(f"Could not find QA band containing {text!r} in {path}.")

def align_mask_to_reference(mask_path: Path, reference_path: Path) -> np.ndarray:
    ref_grid = raster_grid(reference_path)
    out_shape = (int(ref_grid["height"]), int(ref_grid["width"]))
    with rasterio.open(mask_path) as src, rasterio.open(reference_path) as ref:
        arr = src.read(1).astype(np.float32)
        if src.nodata is not None:
            arr[np.isclose(arr, src.nodata)] = np.nan
        if src.width == ref.width and src.height == ref.height and transforms_match(src.transform, ref.transform):
            return np.isfinite(arr) & (arr > 0)
        out = np.full(out_shape, np.nan, dtype=np.float32)
        reproject(
            source=arr,
            destination=out,
            src_transform=src.transform,
            src_crs=crs_to_safe_wkt(src.crs),
            dst_transform=ref.transform,
            dst_crs=crs_to_safe_wkt(ref.crs),
            src_nodata=np.nan,
            dst_nodata=np.nan,
            resampling=Resampling.nearest,
        )
    return np.isfinite(out) & (out > 0)

def candidate_domain_mask(reference_path: Path) -> np.ndarray:
    qa = qa_tif_path()
    ref_grid = raster_grid(reference_path)
    out_shape = (int(ref_grid["height"]), int(ref_grid["width"]))
    if not qa.exists():
        logging.warning("QA file not found. Using all reference pixels as candidate domain.")
        return np.ones(out_shape, dtype=bool)
    band = find_qa_band(qa, "candidate_domain_mask")
    with rasterio.open(qa) as src, rasterio.open(reference_path) as ref:
        arr = src.read(band).astype(np.float32)
        if src.nodata is not None:
            arr[np.isclose(arr, src.nodata)] = np.nan
        if src.width == ref.width and src.height == ref.height and transforms_match(src.transform, ref.transform):
            return np.isfinite(arr) & (arr > 0.5)
        out = np.full(out_shape, np.nan, dtype=np.float32)
        reproject(
            source=arr,
            destination=out,
            src_transform=src.transform,
            src_crs=crs_to_safe_wkt(src.crs),
            dst_transform=ref.transform,
            dst_crs=crs_to_safe_wkt(ref.crs),
            src_nodata=np.nan,
            dst_nodata=np.nan,
            resampling=Resampling.nearest,
        )
    return np.isfinite(out) & (out > 0.5)

def build_offshore_raster(values_at_points: np.ndarray, offshore_points: OffshorePoints) -> np.ndarray:
    grid = offshore_points.grid
    arr = np.full((int(grid["height"]), int(grid["width"])), np.nan, dtype=np.float32)
    arr[offshore_points.row, offshore_points.col] = np.asarray(values_at_points, dtype=np.float32)
    return arr

def write_geotiff(
    path: Path,
    arrays: Sequence[np.ndarray],
    grid: Dict[str, object],
    descriptions: Optional[Sequence[str]] = None,
    nodata: float = NODATA_FLOAT,
) -> None:
    ensure_dir(path.parent)
    crs_safe = grid.get("crs", WGS84_WKT)
    if crs_safe is None or str(crs_safe).upper().startswith("EPSG:"):
        crs_safe = WGS84_WKT
    profile = {
        "driver": "GTiff",
        "height": int(grid["height"]),
        "width": int(grid["width"]),
        "count": len(arrays),
        "dtype": "float32",
        "crs": crs_safe,
        "transform": grid["transform"],
        "nodata": nodata,
        "compress": "deflate",
        "predictor": 2,
        "zlevel": 4,
        "tiled": True,
        "blockxsize": 256,
        "blockysize": 256,
        "BIGTIFF": "IF_SAFER",
    }
    rasterio_env_kwargs = {}
    if PROJ_DATA_DIR is not None:
        rasterio_env_kwargs["PROJ_DATA"] = PROJ_DATA_DIR
    with rasterio.Env(**rasterio_env_kwargs):
        with rasterio.open(path, "w", **profile) as dst:
            for i, arr in enumerate(arrays, start=1):
                out = np.asarray(arr, dtype=np.float32).copy()
                out[~np.isfinite(out)] = nodata
                dst.write(out, i)
                if descriptions is not None and (i - 1) < len(descriptions):
                    dst.set_band_description(i, descriptions[i - 1])
    logging.info("Saved GeoTIFF: %s", path)

def wetland_area_raster_path() -> Path:
    res_label = str(COARSE_RES_DEG).replace(".", "p")
    return INTERMEDIATE_DIR / f"coastal_wetland_area_by_class_{res_label}deg.tif"

def wetland_area_npz_path() -> Path:
    res_label = str(COARSE_RES_DEG).replace(".", "p")
    return INTERMEDIATE_DIR / f"coastal_wetland_area_by_class_{res_label}deg.npz"

def wetland_points_table_path() -> Path:
    res_label = str(COARSE_RES_DEG).replace(".", "p")
    return INTERMEDIATE_DIR / f"coastal_wetland_points_{res_label}deg.parquet"

def cell_area_by_lat_km2(lat_centers_deg: np.ndarray, xres_deg: float, yres_deg: float) -> np.ndarray:
    dlon = np.deg2rad(abs(float(xres_deg)))
    lat1 = np.deg2rad(np.asarray(lat_centers_deg, dtype=np.float64) - abs(float(yres_deg)) / 2.0)
    lat2 = np.deg2rad(np.asarray(lat_centers_deg, dtype=np.float64) + abs(float(yres_deg)) / 2.0)
    return (EARTH_RADIUS_KM ** 2) * dlon * np.abs(np.sin(lat2) - np.sin(lat1))

def source_row_area_km2(src: rasterio.io.DatasetReader) -> np.ndarray:
    lat_centers = src.transform.f + (np.arange(src.height, dtype=np.float64) + 0.5) * src.transform.e
    return cell_area_by_lat_km2(lat_centers, src.transform.a, abs(src.transform.e))

def iter_windows(width: int, height: int, size: int) -> Iterable[Window]:
    for row_off in range(0, height, size):
        h = min(size, height - row_off)
        for col_off in range(0, width, size):
            w = min(size, width - col_off)
            yield Window(col_off=col_off, row_off=row_off, width=w, height=h)

def save_wetland_npz(area_by_class: Dict[int, np.ndarray]) -> None:
    out = wetland_area_npz_path()
    np.savez_compressed(
        out,
        mangrove=area_by_class[1].astype(np.float32),
        salt_marsh=area_by_class[2].astype(np.float32),
        tidal_flat=area_by_class[3].astype(np.float32),
    )
    logging.info("Saved wetland aggregation cache: %s", out)

def load_wetland_npz(path: Path) -> Dict[int, np.ndarray]:
    data = np.load(path)
    return {
        1: data["mangrove"].astype(np.float32),
        2: data["salt_marsh"].astype(np.float32),
        3: data["tidal_flat"].astype(np.float32),
    }

def aggregate_wetland_to_coarse_grid() -> Dict[int, np.ndarray]:
    tif_path = wetland_area_raster_path()
    npz_path = wetland_area_npz_path()
    grid = coarse_grid()
    height = int(grid["height"])
    width = int(grid["width"])

    if npz_path.exists() and not REBUILD_WETLAND_AGGREGATION:
        logging.info("Reusing existing wetland aggregation NPZ cache: %s", npz_path)
        return load_wetland_npz(npz_path)

    if tif_path.exists() and not REBUILD_WETLAND_AGGREGATION:
        logging.info("Reusing existing wetland aggregation GeoTIFF: %s", tif_path)
        out: Dict[int, np.ndarray] = {}
        with rasterio.open(tif_path) as src:
            for class_id in WETLAND_STYLE:
                arr = src.read(class_id).astype(np.float32)
                if src.nodata is not None:
                    arr[np.isclose(arr, src.nodata)] = np.nan
                arr[~np.isfinite(arr)] = 0.0
                out[class_id] = arr
        save_wetland_npz(out)
        return out

    if not WETLAND_PATH.exists():
        raise FileNotFoundError(f"Wetland mask not found: {WETLAND_PATH}")

    logging.info("Aggregating wetland mask to %.3f degree grid.", COARSE_RES_DEG)
    accum = {class_id: np.zeros((height, width), dtype=np.float64) for class_id in WETLAND_STYLE}

    with rasterio.open(WETLAND_PATH) as src:
        row_area = source_row_area_km2(src)
        src_nodata = src.nodata
        windows = list(iter_windows(src.width, src.height, WETLAND_AGGREGATION_WINDOW_SIZE))
        for window in tqdm(windows, desc="Aggregating wetland mask"):
            arr = src.read(1, window=window, masked=False)
            if src_nodata is not None:
                valid = (arr != src_nodata) & np.isin(arr, [1, 2, 3])
            else:
                valid = np.isin(arr, [1, 2, 3])
            if not np.any(valid):
                continue
            rr, cc = np.nonzero(valid)
            vals = arr[rr, cc].astype(np.int16)
            global_rows = rr.astype(np.int64) + int(window.row_off)
            global_cols = cc.astype(np.int64) + int(window.col_off)
            lon = src.transform.c + (global_cols.astype(np.float64) + 0.5) * src.transform.a
            lat = src.transform.f + (global_rows.astype(np.float64) + 0.5) * src.transform.e
            lon = normalize_lon_180(lon)
            target_col = np.floor((lon - COARSE_LON_MIN) / COARSE_RES_DEG).astype(np.int64)
            target_row = np.floor((COARSE_LAT_MAX - lat) / COARSE_RES_DEG).astype(np.int64)
            inside = (
                (target_col >= 0) & (target_col < width) &
                (target_row >= 0) & (target_row < height) &
                (lat >= COARSE_LAT_MIN) & (lat <= COARSE_LAT_MAX)
            )
            if inside.sum() == 0:
                continue
            target_row = target_row[inside]
            target_col = target_col[inside]
            vals = vals[inside]
            area = row_area[global_rows[inside]].astype(np.float64)
            linear = target_row * width + target_col
            for class_id in WETLAND_STYLE:
                sel = vals == class_id
                if np.any(sel):
                    binc = np.bincount(linear[sel], weights=area[sel], minlength=height * width)
                    accum[class_id] += binc.reshape((height, width))
            del arr, valid, rr, cc, vals, global_rows, global_cols, lon, lat, target_row, target_col, area, linear
            gc.collect()

    out = {class_id: accum[class_id].astype(np.float32) for class_id in WETLAND_STYLE}
    save_wetland_npz(out)
    try:
        write_geotiff(
            tif_path,
            [out[1], out[2], out[3]],
            grid,
            descriptions=["Mangrove area (km2)", "Salt marsh area (km2)", "Tidal flat area (km2)"],
        )
    except Exception as exc:
        logging.warning("GeoTIFF wetland aggregation write failed, but NPZ cache is saved. Error: %s", exc)
    return out

def build_wetland_points(area_by_class_grid: Dict[int, np.ndarray]) -> WetlandPoints:
    grid = coarse_grid()
    height = int(grid["height"])
    width = int(grid["width"])
    total = np.zeros((height, width), dtype=np.float32)
    for arr in area_by_class_grid.values():
        total += np.asarray(arr, dtype=np.float32)
    valid = np.isfinite(total) & (total > 0)
    row, col = np.nonzero(valid)
    lon = np.asarray(grid["lon"], dtype=np.float64)[col]
    lat = np.asarray(grid["lat"], dtype=np.float64)[row]
    linear = row.astype(np.int64) * width + col.astype(np.int64)
    area_by_class = {
        class_id: np.asarray(area_by_class_grid[class_id][row, col], dtype=np.float32)
        for class_id in WETLAND_STYLE
    }
    total_area = np.asarray(total[row, col], dtype=np.float32)
    point_df = pd.DataFrame(
        {
            "row": row,
            "col": col,
            "linear_index": linear,
            "lon": lon,
            "lat": lat,
            "total_wetland_area_km2": total_area,
            "mangrove_area_km2": area_by_class[1],
            "salt_marsh_area_km2": area_by_class[2],
            "tidal_flat_area_km2": area_by_class[3],
        }
    )
    try:
        point_df.to_parquet(wetland_points_table_path(), index=False)
    except Exception:
        point_df.to_csv(wetland_points_table_path().with_suffix(".csv"), index=False)
    logging.info("Coarse wetland receptor cells: %d", len(lon))
    for class_id, style in WETLAND_STYLE.items():
        logging.info(
            "  %-12s area: %.2f km2 in %d coarse cells",
            style["name"],
            float(np.nansum(area_by_class[class_id])),
            int(np.sum(area_by_class[class_id] > 0)),
        )
    return WetlandPoints(
        lon=lon, lat=lat, row=row, col=col, linear_index=linear,
        area_by_class=area_by_class, total_area=total_area,
    )

def discover_analysis_cases() -> List[AnalysisCase]:
    cases: List[AnalysisCase] = []
    for wave_case in ANALYSIS_WAVE_CASES:
        path = pct_tif_path(wave_case)
        if not path.exists():
            logging.warning("Missing percent-change file for wave_case=%s. Skipping.", wave_case)
            continue
        meta = read_band_metadata(path)
        if meta.empty:
            continue
        meta["slr_version"] = meta["slr_version"].astype(str)
        meta["slr_scenario"] = meta["slr_scenario"].astype(str)
        meta["slr_quantile_label"] = meta["slr_quantile_label"].astype(str)
        meta["year"] = pd.to_numeric(meta["slr_target_year"], errors="coerce")
        meta = meta[np.isfinite(meta["year"])].copy()
        meta["year"] = meta["year"].astype(int)
        meta["quantile_canonical"] = meta["slr_quantile_label"].map(canonical_quantile_label)
        meta = meta[
            meta["slr_version"].isin(SELECTED_VERSIONS) &
            meta["slr_scenario"].isin(SELECTED_SCENARIOS) &
            meta["year"].isin(SELECTED_YEARS) &
            meta["quantile_canonical"].isin(SELECTED_CANONICAL_QUANTILES)
        ].copy()
        if meta.empty:
            logging.warning("No selected bands found for wave_case=%s after filtering.", wave_case)
            continue
        for row in meta.itertuples(index=False):
            year = int(row.year)
            qcanon = str(row.quantile_canonical)
            qraw = str(row.slr_quantile_label)
            cid = case_key(wave_case, str(row.slr_version), str(row.slr_scenario), year, qcanon)
            cases.append(
                AnalysisCase(
                    case_id=cid,
                    wave_case=wave_case,
                    version=str(row.slr_version),
                    scenario=str(row.slr_scenario),
                    year=year,
                    quantile_raw=qraw,
                    quantile_canonical=qcanon,
                    band=int(row.band),
                    path=path,
                )
            )
    if len(cases) == 0:
        raise RuntimeError("No valid analysis cases were discovered. Check wave-case files and metadata tags.")
    case_df = pd.DataFrame([c.__dict__ for c in cases]).sort_values(
        ["wave_case", "version", "scenario", "year", "quantile_canonical"]
    )
    case_df.to_csv(TABLE_DIR / "analysis_cases.csv", index=False)
    logging.info("Analysis cases discovered: %d", len(cases))
    logging.info("Wave cases included: %s", sorted(case_df["wave_case"].unique().tolist()))
    return cases

def build_offshore_points(cases: List[AnalysisCase]) -> OffshorePoints:
    reference_path = cases[0].path
    grid = raster_grid(reference_path)
    height = int(grid["height"])
    width = int(grid["width"])
    final_mask = align_mask_to_reference(FINAL_MASK_PATH, reference_path)
    domain_mask = candidate_domain_mask(reference_path)
    base_mask = final_mask & domain_mask
    case_values_full: List[np.ndarray] = []
    union_valid = np.zeros((height, width), dtype=bool)
    logging.info("Reading offshore shear-change bands.")
    for case in tqdm(cases, desc="Reading cases"):
        case_grid = raster_grid(case.path)
        if int(case_grid["height"]) != height or int(case_grid["width"]) != width:
            raise ValueError(f"Case grid size differs from the reference grid: {case.path}")
        arr = read_band(case.path, case.band)
        arr = np.where(base_mask & np.isfinite(arr), arr, np.nan).astype(np.float32)
        union_valid |= np.isfinite(arr)
        case_values_full.append(arr)
    row, col = np.nonzero(union_valid)
    linear = row.astype(np.int64) * width + col.astype(np.int64)
    lon = normalize_lon_180(np.asarray(grid["lon"], dtype=np.float64)[col])
    lat = np.asarray(grid["lat"], dtype=np.float64)[row]
    values = np.column_stack([arr[row, col] for arr in case_values_full]).astype(np.float32)
    case_ids = [c.case_id for c in cases]
    logging.info("Offshore coastal-ring pixels in union mask: %d", len(lon))
    return OffshorePoints(
        lon=lon, lat=lat, row=row, col=col, linear_index=linear,
        case_values=values, case_ids=case_ids, grid=grid,
    )

def connection_table_path(max_dmax_km: float) -> Path:
    res_label = str(COARSE_RES_DEG).replace(".", "p")
    d_label = str(int(round(max_dmax_km)))
    return INTERMEDIATE_DIR / f"offshore_to_wetland_neighbors_coarse{res_label}_maxD{d_label}km.npz"

def load_or_build_connection_table(
    wetland_points: WetlandPoints,
    offshore_points: OffshorePoints,
    max_dmax_km: float,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    out_path = connection_table_path(max_dmax_km)
    if out_path.exists() and not REBUILD_CONNECTION_TABLE:
        logging.info("Reusing connection table: %s", out_path)
        data = np.load(out_path)
        return data["row"], data["col"], data["dist_km"]
    logging.info("Building offshore-to-wetland connection table up to Dmax=%.1f km", max_dmax_km)
    off_xyz = lonlat_to_xyz(offshore_points.lon, offshore_points.lat)
    tree = cKDTree(off_xyz)
    max_chord = 2.0 * np.sin(max_dmax_km / (2.0 * EARTH_RADIUS_KM))
    row_list: List[np.ndarray] = []
    col_list: List[np.ndarray] = []
    dist_list: List[np.ndarray] = []
    n_wet = len(wetland_points.lon)
    for start in tqdm(range(0, n_wet, DISTANCE_CHUNK_SIZE), desc="Building connections"):
        end = min(start + DISTANCE_CHUNK_SIZE, n_wet)
        xyz_chunk = lonlat_to_xyz(wetland_points.lon[start:end], wetland_points.lat[start:end])
        try:
            neighbors = tree.query_ball_point(xyz_chunk, r=max_chord, workers=MAX_CORES)
        except TypeError:
            neighbors = tree.query_ball_point(xyz_chunk, r=max_chord)
        for local_j, nbr in enumerate(neighbors):
            if len(nbr) == 0:
                continue
            j = start + local_j
            ii = np.asarray(nbr, dtype=np.int32)
            d = haversine_km(
                wetland_points.lon[j],
                wetland_points.lat[j],
                offshore_points.lon[ii],
                offshore_points.lat[ii],
            )
            keep = np.isfinite(d) & (d <= max_dmax_km)
            if not np.any(keep):
                continue
            ii = ii[keep]
            d = d[keep].astype(np.float32)
            row_list.append(ii)
            col_list.append(np.full(ii.shape, j, dtype=np.int32))
            dist_list.append(d)
    if len(row_list) == 0:
        raise RuntimeError("No offshore-wetland neighbor pairs were found.")
    row = np.concatenate(row_list)
    col = np.concatenate(col_list)
    dist_km = np.concatenate(dist_list)
    np.savez_compressed(out_path, row=row, col=col, dist_km=dist_km)
    logging.info("Saved connection table: %s", out_path)
    return row, col, dist_km

def build_kernel_load_result(
    row: np.ndarray,
    col: np.ndarray,
    dist_km: np.ndarray,
    wetland_points: WetlandPoints,
    offshore_points: OffshorePoints,
    kernel_spec: Dict[str, object],
    wetland_domain_mask: Optional[np.ndarray] = None,
    offshore_domain_mask: Optional[np.ndarray] = None,
) -> KernelLoadResult:
    lambda_km = float(kernel_spec["lambda_km"])
    dmax_km = float(kernel_spec["dmax_km"])
    label = str(kernel_spec["label"])
    sel = np.isfinite(dist_km) & (dist_km <= dmax_km)

    # Apply the domain masks at the connection-pair level. This allows the same
    # global neighbor table to be reused for both the complete-domain benchmark
    # and the primary <=66N manuscript analysis without rebuilding the KD-tree.
    if wetland_domain_mask is not None:
        wetland_domain_mask = np.asarray(wetland_domain_mask, dtype=bool)
        sel &= wetland_domain_mask[col]
    if offshore_domain_mask is not None:
        offshore_domain_mask = np.asarray(offshore_domain_mask, dtype=bool)
        sel &= offshore_domain_mask[row]

    if sel.sum() == 0:
        raise RuntimeError(f"No valid neighbor pairs remain for kernel={label}.")
    weights = np.exp(-dist_km[sel] / lambda_km).astype(np.float32)
    mat = csr_matrix(
        (weights, (row[sel], col[sel])),
        shape=(len(offshore_points.lon), len(wetland_points.lon)),
        dtype=np.float32,
    )
    mat.sum_duplicates()
    total_load = np.asarray(mat.dot(wetland_points.total_area), dtype=np.float32)
    class_loads = {
        class_id: np.asarray(mat.dot(wetland_points.area_by_class[class_id]), dtype=np.float32)
        for class_id in WETLAND_STYLE
    }
    return KernelLoadResult(
        label=label, lambda_km=lambda_km, dmax_km=dmax_km,
        total_load=total_load, class_loads=class_loads,
    )

def compute_case_summary_for_kernel(
    kernel_load: KernelLoadResult,
    offshore_points: OffshorePoints,
    wetland_points: WetlandPoints,
    wetland_domain_mask: Optional[np.ndarray] = None,
) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for j, cid in enumerate(offshore_points.case_ids):
        meta = parse_case_id(cid)
        delta = offshore_points.case_values[:, j].astype(np.float64)
        negative = np.clip(-delta, 0, None)
        for class_id, style in WETLAND_STYLE.items():
            load = kernel_load.class_loads[class_id].astype(np.float64)
            valid = np.isfinite(delta) & np.isfinite(load) & (load > 0)
            row: Dict[str, object] = {
                "kernel_label": kernel_load.label,
                "lambda_km": kernel_load.lambda_km,
                "dmax_km": kernel_load.dmax_km,
                "wetland_class_id": class_id,
                "wetland_class": style["name"],
                "wetland_class_pretty": style["pretty"],
                "source_total_wetland_area_km2": float(
                    np.nansum(
                        wetland_points.area_by_class[class_id]
                        if wetland_domain_mask is None
                        else wetland_points.area_by_class[class_id][
                            np.asarray(wetland_domain_mask, dtype=bool)
                        ]
                    )
                ),
                **meta,
            }
            summary = weighted_summary(delta[valid], load[valid])
            row.update({
                "delta_n": summary["n"],
                "delta_weight_sum": summary["weight_sum"],
                "delta_mean": summary["mean"],
                "delta_p05": summary["p05"],
                "delta_p25": summary["p25"],
                "delta_median": summary["median"],
                "delta_p75": summary["p75"],
                "delta_p95": summary["p95"],
                "delta_min": summary["min"],
                "delta_max": summary["max"],
            })
            row["linked_load_sum"] = float(np.sum(load[valid])) if valid.any() else np.nan
            add_fraction_metrics(row, delta[valid], load[valid])
            severity = load[valid] * negative[valid]
            row["total_severity"] = float(np.sum(severity)) if severity.size > 0 else np.nan
            row["mean_severity_per_linked_load"] = (
                float(np.sum(severity) / np.sum(load[valid]))
                if valid.any() and np.sum(load[valid]) > 0 else np.nan
            )
            row.update(top_fraction_share(severity, fractions=(0.01, 0.05, 0.10)))
            rows.append(row)
    return pd.DataFrame(rows)

def compute_relative_severity(summary_df: pd.DataFrame) -> pd.DataFrame:
    df = summary_df.copy()
    main = df[
        (df["kernel_label"] == MAIN_KERNEL_LABEL) &
        (df["wave_case"] == MAIN_WAVE_CASE) &
        (df["version"] == MAIN_VERSION) &
        (df["scenario"] == MAIN_SCENARIO) &
        (df["year"] == MAIN_YEAR) &
        (df["quantile_canonical"] == MAIN_QUANTILE)
    ][["wetland_class_id", "total_severity"]].copy()
    main = main.rename(columns={"total_severity": "main_total_severity"})
    df = df.merge(main, on="wetland_class_id", how="left")
    df["severity_rel_to_main_pct"] = 100.0 * df["total_severity"] / df["main_total_severity"]
    return df

def make_pct_cmap() -> LinearSegmentedColormap:
    colors = ["#0b2c6b", "#2b6cb0", "#6f73b8", "#c49a84", "#f7f2ea"]
    cmap = LinearSegmentedColormap.from_list("lifeline_weakening", colors, N=256)
    cmap.set_under("#081a4d")
    cmap.set_over("#b2182b")
    return cmap

def make_frac_cmap() -> LinearSegmentedColormap:
    return LinearSegmentedColormap.from_list(
        "fraction_cmap",
        ["#f7fbff", "#c6dbef", "#6baed6", "#2171b5", "#08306b"],
        N=256,
    )

def make_severity_cmap() -> LinearSegmentedColormap:
    return LinearSegmentedColormap.from_list(
        "severity_cmap",
        ["#fff5eb", "#fdd0a2", "#f16913", "#b30000", "#49006a"],
        N=256,
    )

def draw_map_background(ax) -> None:
    if HAS_CARTOPY and hasattr(ax, "projection"):
        try:
            ax.add_feature(cfeature.LAND, facecolor="0.93", edgecolor="none", zorder=0)
        except Exception:
            pass
        try:
            ax.coastlines(linewidth=0.32, color="0.25", zorder=3)
        except Exception:
            pass
        ax.set_global()
    else:
        ax.set_xlim(-180, 180)
        ax.set_ylim(-90, 90)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

def scatter_global_points(ax, lon: np.ndarray, lat: np.ndarray, values: np.ndarray, sizes: np.ndarray, cmap, norm):
    if HAS_CARTOPY and hasattr(ax, "projection"):
        return ax.scatter(
            lon, lat, c=values, s=sizes, cmap=cmap, norm=norm,
            edgecolors="none", linewidths=0.0, alpha=MAP_ALPHA,
            transform=ccrs.PlateCarree(), rasterized=True, zorder=2,
        )
    return ax.scatter(
        lon, lat, c=values, s=sizes, cmap=cmap, norm=norm,
        edgecolors="none", linewidths=0.0, alpha=MAP_ALPHA,
        rasterized=True, zorder=2,
    )

def marker_size_bin_edges(reference_load: np.ndarray) -> Tuple[float, float]:
    ref = np.asarray(reference_load, dtype=float)
    ok = np.isfinite(ref) & (ref > 0)
    if ok.sum() == 0:
        return 0.0, 1.0
    logv = np.log10(ref[ok] + 1.0)
    e1, e2 = np.nanpercentile(logv, MAP_SIZE_BIN_EDGE_QUANTILES)
    if not np.isfinite(e1) or not np.isfinite(e2) or e2 <= e1:
        e1, e2 = np.nanpercentile(logv, [33.0, 67.0])
    if not np.isfinite(e1) or not np.isfinite(e2) or e2 <= e1:
        e1, e2 = 0.0, 1.0
    return float(e1), float(e2)

def marker_size_class_index(load: np.ndarray, reference_load: Optional[np.ndarray] = None) -> np.ndarray:
    load = np.asarray(load, dtype=float)
    ref = load if reference_load is None else np.asarray(reference_load, dtype=float)
    cls = np.full(load.shape, -1, dtype=np.int16)
    ok = np.isfinite(load) & (load > 0)
    if ok.sum() == 0:
        return cls
    e1, e2 = marker_size_bin_edges(ref)
    logv = np.log10(load[ok] + 1.0)
    cls_ok = np.zeros(logv.shape, dtype=np.int16)
    cls_ok[logv > e1] = 1
    cls_ok[logv > e2] = 2
    cls[ok] = cls_ok
    return cls

def marker_sizes_from_load(load: np.ndarray, reference_load: Optional[np.ndarray] = None) -> np.ndarray:
    load = np.asarray(load, dtype=float)
    ref = load if reference_load is None else np.asarray(reference_load, dtype=float)
    sizes = np.full(load.shape, float(MAP_SIZE_LEVEL_AREAS[0]), dtype=float)
    cls = marker_size_class_index(load, ref)
    for i, area in enumerate(MAP_SIZE_LEVEL_AREAS):
        sizes[cls == i] = float(area)
    sizes[cls < 0] = float(MAP_SIZE_LEVEL_AREAS[0])
    return sizes

def round_to_nice_number(x: float) -> float:
    if not np.isfinite(x) or x <= 0:
        return np.nan
    power = 10 ** math.floor(math.log10(x))
    base = x / power
    if base < 1.5:
        nice = 1.0
    elif base < 3.5:
        nice = 2.0
    elif base < 7.5:
        nice = 5.0
    else:
        nice = 10.0
    return nice * power

def format_load_label(v: float) -> str:
    if not np.isfinite(v):
        return "n/a"
    if v >= 1000:
        return f"{int(round(v)):,}"
    if v >= 10:
        return f"{int(round(v))}"
    return f"{v:.1f}"

def size_legend_values(load: np.ndarray) -> Tuple[np.ndarray, List[str], np.ndarray]:
    load = np.asarray(load, dtype=float)
    pos = load[np.isfinite(load) & (load > 0)]
    if pos.size == 0:
        vals = np.array([1.0, 10.0, 100.0], dtype=float)
    else:
        vals = np.nanpercentile(pos, MAP_SIZE_LEGEND_QUANTILES).astype(float)
        rounded = np.array([round_to_nice_number(v) for v in vals], dtype=float)
        if np.all(np.isfinite(rounded)) and np.all(np.diff(rounded) > 0):
            vals = rounded
    vals = np.clip(vals, 1e-6, None)
    labels = [format_load_label(v) for v in vals]
    sizes = np.asarray(MAP_SIZE_LEVEL_AREAS, dtype=float)
    return vals, labels, sizes

def add_size_legend(ax, load: np.ndarray) -> None:
    _, labels, sizes = size_legend_values(load)
    handles = [
        Line2D(
            [0], [0], marker="o", linestyle="None",
            markerfacecolor="0.55", markeredgecolor="0.30", markeredgewidth=0.35,
            markersize=math.sqrt(float(s)), label=lab, alpha=0.95,
        )
        for s, lab in zip(sizes, labels)
    ]
    leg = ax.legend(
        handles=handles, title="Impacted wetland area", loc="lower left",
        bbox_to_anchor=(0.415, 0.020), frameon=True, fancybox=False,
        borderpad=0.35, handletextpad=0.9, labelspacing=0.55, borderaxespad=0.0,
    )
    leg.get_frame().set_linewidth(0.5)
    leg.get_frame().set_alpha(0.95)
    if leg.get_title() is not None:
        leg.get_title().set_fontsize(SMALL_FONT)

def gaussian_smooth_1d(y: np.ndarray, sigma_bins: float) -> np.ndarray:
    sigma_bins = float(max(0.5, sigma_bins))
    radius = int(max(3, math.ceil(sigma_bins * 4)))
    x = np.arange(-radius, radius + 1)
    k = np.exp(-0.5 * (x / sigma_bins) ** 2)
    k /= np.sum(k)
    return np.convolve(y, k, mode="same")

def draw_weighted_violin(ax, xpos: float, values: np.ndarray, weights: np.ndarray, color: str) -> None:
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    ok = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if ok.sum() < 5:
        return
    v = values[ok]
    w = weights[ok]
    qlo, qhi = weighted_quantile(v, w, VIOLIN_RANGE_QUANTILES)
    if not np.isfinite(qlo) or not np.isfinite(qhi) or qhi <= qlo:
        qlo = float(np.nanmin(v))
        qhi = float(np.nanmax(v))
    vrange = max(qhi - qlo, 1e-6)
    pad = vrange * VIOLIN_TAIL_PAD_FRACTION
    ymin = qlo - pad
    ymax = qhi + pad
    edges = np.linspace(ymin, ymax, VIOLIN_BINS + 1)
    hist, _ = np.histogram(v, bins=edges, weights=w)
    centers = 0.5 * (edges[:-1] + edges[1:])
    density = hist.astype(np.float64)
    if np.sum(density) <= 0:
        return
    density = gaussian_smooth_1d(density, VIOLIN_SMOOTH_SIGMA_BINS)
    density = np.clip(density, 0, None)
    dense_y = np.linspace(centers[0], centers[-1], VIOLIN_DENSE_POINTS)
    dense_d = np.interp(dense_y, centers, density)
    if np.nanmax(dense_d) > 0:
        dense_d = dense_d / np.nanmax(dense_d)
    half_width = VIOLIN_WIDTH * dense_d
    left = xpos - half_width
    right = xpos + half_width
    ax.fill_betweenx(
        dense_y, left, right,
        facecolor=color, edgecolor=VIOLIN_EDGE_COLOR,
        linewidth=VIOLIN_EDGE_WIDTH, alpha=VIOLIN_ALPHA, zorder=2,
    )
    q25, q50, q75 = weighted_quantile(v, w, [0.25, 0.50, 0.75])
    ax.plot([xpos, xpos], [q25, q75], color=SUMMARY_WHISKER_COLOR, lw=1.3, zorder=4)
    ax.plot([xpos - 0.14, xpos + 0.14], [q50, q50], color=SUMMARY_LINE_COLOR, lw=1.8, zorder=5)

def annotate_panel(ax, text: str, x: float = 0.0, y: float = 1.02) -> None:
    ax.text(
        x, y, text, transform=ax.transAxes,
        fontsize=PANEL_FONT, fontweight="bold",
        ha="left", va="bottom",
    )

def heatmap_panel(
    ax,
    mat: np.ndarray,
    row_labels: List[str],
    col_labels: List[str],
    cmap,
    vmin: Optional[float],
    vmax: Optional[float],
    cbar_label: str,
    value_format: str = "{:.1f}",
) -> None:
    im = ax.imshow(mat, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(len(col_labels)))
    ax.set_xticklabels(col_labels)
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            if np.isfinite(val):
                threshold = (vmin + vmax) / 2.0 if vmin is not None and vmax is not None else np.nanmedian(mat)
                text_color = "white" if np.isfinite(threshold) and val > threshold else "black"
                ax.text(j, i, value_format.format(val), ha="center", va="center", fontsize=TINY_FONT, color=text_color)
    cbar = plt.colorbar(im, ax=ax, fraction=0.040, pad=0.02)
    cbar.set_label(cbar_label)

def require_case_index(offshore_points: OffshorePoints, cid: str) -> int:
    if cid not in offshore_points.case_ids:
        available = "\n".join(offshore_points.case_ids[:20])
        raise KeyError(f"Required case was not found: {cid}\nFirst available cases:\n{available}")
    return offshore_points.case_ids.index(cid)

def make_main_weakening_figure(
    offshore_points: OffshorePoints,
    kernel_load: KernelLoadResult,
    summary_df: pd.DataFrame,
) -> None:
    cid = main_case_id()
    case_idx = require_case_index(offshore_points, cid)
    delta = offshore_points.case_values[:, case_idx].astype(np.float64)
    total_load = kernel_load.total_load.astype(np.float64)
    class_loads = kernel_load.class_loads
    valid_map = np.isfinite(delta) & np.isfinite(total_load) & (total_load > 0)
    fig = plt.figure(figsize=FIG1_SIZE)

    if HAS_CARTOPY:
        axa = fig.add_axes(
            [FIG1_PANEL_A_LEFT, FIG1_PANEL_A_BOTTOM, FIG1_PANEL_A_WIDTH, FIG1_PANEL_A_HEIGHT],
            projection=ccrs.Robinson(),
        )
    else:
        axa = fig.add_axes([FIG1_PANEL_A_LEFT, FIG1_PANEL_A_BOTTOM, FIG1_PANEL_A_WIDTH, FIG1_PANEL_A_HEIGHT])
    draw_map_background(axa)
    cmap = make_pct_cmap()
    norm = Normalize(vmin=PCT_VMIN, vmax=PCT_VMAX)
    sizes = marker_sizes_from_load(total_load[valid_map], reference_load=total_load[valid_map])
    scatter_global_points(
        axa,
        offshore_points.lon[valid_map],
        offshore_points.lat[valid_map],
        delta[valid_map],
        sizes,
        cmap=cmap,
        norm=norm,
    )
    annotate_panel(axa, "a")
    add_size_legend(axa, total_load[valid_map])
    cax = fig.add_axes([FIG1_CBAR_LEFT, FIG1_CBAR_BOTTOM, FIG1_CBAR_WIDTH, FIG1_CBAR_HEIGHT])
    cb = plt.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=cax, orientation="horizontal")
    cb.set_label(r"$	au_{\mathrm{wave}}$ change (%)")
    cb.set_ticks(PCT_TICKS)

    axb = fig.add_axes([FIG1_PANEL_B_LEFT, FIG1_SECOND_ROW_BOTTOM, FIG1_PANEL_B_WIDTH, FIG1_SECOND_ROW_HEIGHT])
    for i, class_id in enumerate([1, 2, 3], start=1):
        draw_weighted_violin(
            axb,
            i,
            delta,
            class_loads[class_id],
            color=WETLAND_STYLE[class_id]["color"],
        )
    axb.axhline(0, color="0.75", lw=0.8, zorder=0)
    axb.set_xlim(0.45, 3.55)
    axb.set_xticks([1, 2, 3])
    axb.set_xticklabels([WETLAND_STYLE[c]["pretty"] for c in [1, 2, 3]])
    axb.set_ylabel(r"$	au_{\mathrm{wave}}$ change (%)")
    annotate_panel(axb, "b")

    axc = fig.add_axes([FIG1_PANEL_C_LEFT, FIG1_SECOND_ROW_BOTTOM, FIG1_PANEL_C_WIDTH, FIG1_SECOND_ROW_HEIGHT])
    sub = summary_df[
        (summary_df["kernel_label"] == kernel_load.label) &
        (summary_df["wave_case"] == MAIN_WAVE_CASE) &
        (summary_df["version"] == MAIN_VERSION) &
        (summary_df["year"] == MAIN_YEAR) &
        (summary_df["quantile_canonical"] == MAIN_QUANTILE) &
        (summary_df["scenario"].isin(SELECTED_SCENARIOS))
    ].copy()
    scenario_order = SELECTED_SCENARIOS
    x = np.arange(len(scenario_order))
    width = 0.22
    offsets = [-width, 0.0, width]
    for offset, class_id in zip(offsets, [1, 2, 3]):
        vals = []
        for s in scenario_order:
            sel = sub[(sub["scenario"] == s) & (sub["wetland_class_id"] == class_id)]
            vals.append(100.0 * float(sel["frac_linked_to_gt25_decrease"].iloc[0]) if len(sel) > 0 else np.nan)
        axc.bar(
            x + offset,
            vals,
            width=width,
            color=WETLAND_STYLE[class_id]["color"],
            edgecolor="0.2",
            linewidth=0.5,
            label=WETLAND_STYLE[class_id]["pretty"],
        )
    axc.set_xticks(x)
    axc.set_xticklabels([scenario_label(s) for s in scenario_order])
    axc.set_ylabel(r"Wetlands linked to >25% $	au_{\mathrm{wave}}$ decrease (%)")
    axc.legend(frameon=False, loc="upper left")
    annotate_panel(axc, "c")
    save_jpg(fig, FIGURE_DIR / "Figure1_ocean_lifeline_weakening_main.jpg")

def make_main_severity_figure(
    offshore_points: OffshorePoints,
    kernel_load: KernelLoadResult,
    summary_df: pd.DataFrame,
) -> None:
    cid = main_case_id()
    case_idx = require_case_index(offshore_points, cid)
    delta = offshore_points.case_values[:, case_idx].astype(np.float64)
    # Keep cached/table values in their original percentage-based units.
    # For this figure, use fractional stress loss (30% = 0.30), yielding km^2.
    negative = np.clip(-delta, 0, None) / 100.0
    severity_by_class = {
        class_id: kernel_load.class_loads[class_id].astype(np.float64) * negative
        for class_id in [1, 2, 3]
    }
    positive_list = [v[np.isfinite(v) & (v > 0)] for v in severity_by_class.values()]
    positive_list = [v for v in positive_list if v.size > 0]
    if len(positive_list) == 0:
        raise RuntimeError("No positive severity values exist for the main case.")
    positive_vals = np.concatenate(positive_list)
    sev_plot_vals = np.log10(positive_vals + 1.0)
    vmin = float(np.nanpercentile(sev_plot_vals, 5))
    vmax = float(np.nanpercentile(sev_plot_vals, 99.5))
    if not np.isfinite(vmax) or vmax <= vmin:
        vmin, vmax = 0.0, 1.0
    cmap = make_severity_cmap()
    norm = Normalize(vmin=vmin, vmax=vmax)
    fig = plt.figure(figsize=FIG2_SIZE)
    map_positions = [
        [0.04, 0.55, 0.29, 0.34],
        [0.355, 0.55, 0.29, 0.34],
        [0.67, 0.55, 0.29, 0.34],
    ]
    for pos, class_id, panel_letter in zip(map_positions, [1, 2, 3], ["a", "b", "c"]):
        if HAS_CARTOPY:
            ax = fig.add_axes(pos, projection=ccrs.Robinson())
        else:
            ax = fig.add_axes(pos)
        draw_map_background(ax)
        sev = severity_by_class[class_id]
        valid = np.isfinite(sev) & (sev > 0)
        scatter_global_points(
            ax,
            offshore_points.lon[valid],
            offshore_points.lat[valid],
            np.log10(sev[valid] + 1.0),
            np.full(valid.sum(), 6.0),
            cmap=cmap,
            norm=norm,
        )
        annotate_panel(ax, panel_letter)
        ax.text(
            0.02, 0.02, WETLAND_STYLE[class_id]["pretty"],
            transform=ax.transAxes,
            ha="left", va="bottom",
            fontsize=SMALL_FONT,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=2.2),
        )
    cax = fig.add_axes([0.32, 0.51, 0.36, 0.020])
    cb = plt.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=cax, orientation="horizontal")
    # A_sw denotes stress-weighted wetland area in km^2.
    cb.set_label("Stress-weighted wetland area\n"
                 r"$\log_{10}(1 + A_{\mathrm{sw}} / \mathrm{km}^{2})$")
    axd = fig.add_axes([0.10, 0.10, 0.82, 0.25])
    sub = summary_df[
        (summary_df["kernel_label"] == kernel_load.label) &
        (summary_df["wave_case"] == MAIN_WAVE_CASE) &
        (summary_df["version"] == MAIN_VERSION) &
        (summary_df["scenario"] == MAIN_SCENARIO) &
        (summary_df["year"] == MAIN_YEAR) &
        (summary_df["quantile_canonical"] == MAIN_QUANTILE)
    ].copy().sort_values("wetland_class_id")
    x = np.arange(3)
    vals = []
    for class_id in [1, 2, 3]:
        sel = sub[sub["wetland_class_id"] == class_id]
        vals.append(float(sel["total_severity"].iloc[0]) / 100.0 if len(sel) > 0 else np.nan)
    vals = np.asarray(vals, dtype=float)
    colors = [WETLAND_STYLE[c]["color"] for c in [1, 2, 3]]
    axd.bar(x, vals, color=colors, edgecolor="0.2", linewidth=0.5)
    axd.set_xticks(x)
    axd.set_xticklabels([WETLAND_STYLE[c]["pretty"] for c in [1, 2, 3]])
    axd.set_ylabel("Stress-weighted wetland area\n" + r"(km$^2$)")
    annotate_panel(axd, "d")
    save_jpg(fig, FIGURE_DIR / "Figure2_ocean_lifeline_severity_main.jpg")

def heatmap_panel(
    ax,
    mat: np.ndarray,
    row_labels: List[str],
    col_labels: List[str],
    cmap,
    vmin: Optional[float],
    vmax: Optional[float],
    cbar_label: str,
    value_format: str = "{:.1f}",
) -> None:
    im = ax.imshow(mat, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(len(col_labels)))
    ax.set_xticklabels(col_labels)
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            if np.isfinite(val):
                threshold = (vmin + vmax) / 2.0 if vmin is not None and vmax is not None else np.nanmedian(mat)
                text_color = "white" if np.isfinite(threshold) and val > threshold else "black"
                ax.text(j, i, value_format.format(val), ha="center", va="center", fontsize=TINY_FONT, color=text_color)
    cbar = plt.colorbar(im, ax=ax, fraction=0.040, pad=0.02)
    cbar.set_label(cbar_label)

def make_scenario_sensitivity_tau_figure(summary_df: pd.DataFrame) -> None:
    sub = summary_df[
        (summary_df["kernel_label"] == MAIN_KERNEL_LABEL) &
        (summary_df["wave_case"] == MAIN_WAVE_CASE) &
        (summary_df["version"] == MAIN_VERSION) &
        (summary_df["quantile_canonical"] == MAIN_QUANTILE) &
        (summary_df["scenario"].isin(SELECTED_SCENARIOS)) &
        (summary_df["year"].isin(SELECTED_YEARS))
    ].copy()
    row_keys = []
    row_labels = []
    for year in SELECTED_YEARS:
        for scenario in SELECTED_SCENARIOS:
            row_keys.append((scenario, year))
            row_labels.append(f"{year} | {scenario_label(scenario)}")
    col_ids = [1, 2, 3]
    col_labels = [WETLAND_STYLE[c]["pretty"] for c in col_ids]
    mat_mean = np.full((len(row_keys), len(col_ids)), np.nan)
    mat_frac = np.full((len(row_keys), len(col_ids)), np.nan)
    for i, (scenario, year) in enumerate(row_keys):
        for j, class_id in enumerate(col_ids):
            sel = sub[(sub["scenario"] == scenario) & (sub["year"] == year) & (sub["wetland_class_id"] == class_id)]
            if len(sel) > 0:
                mat_mean[i, j] = float(sel["delta_mean"].iloc[0])
                mat_frac[i, j] = 100.0 * float(sel["frac_linked_to_gt25_decrease"].iloc[0])
    fig, axes = plt.subplots(1, 2, figsize=HEATMAP_FIGSIZE)
    heatmap_panel(axes[0], mat_mean, row_labels, col_labels, cmap=make_pct_cmap(), vmin=-30, vmax=0, cbar_label=r"Mean $	au_{\mathrm{wave}}$ change (%)")
    annotate_panel(axes[0], "a")
    heatmap_panel(axes[1], mat_frac, row_labels, col_labels, cmap=make_frac_cmap(), vmin=0, vmax=max(5.0, safe_nanmax(mat_frac, fallback=5.0)), cbar_label=r"Wetlands linked to >25% $	au_{\mathrm{wave}}$ decrease (%)")
    annotate_panel(axes[1], "b")
    save_jpg(fig, FIGURE_DIR / "Sensitivity1_scenario_year_tau.jpg")

def make_wave_quantile_sensitivity_tau_figure(summary_df: pd.DataFrame) -> None:
    available_wave_cases = [w for w in ANALYSIS_WAVE_CASES if w in set(summary_df["wave_case"].unique())]
    sub = summary_df[
        (summary_df["kernel_label"] == MAIN_KERNEL_LABEL) &
        (summary_df["version"] == MAIN_VERSION) &
        (summary_df["scenario"] == MAIN_SCENARIO) &
        (summary_df["year"] == MAIN_YEAR) &
        (summary_df["quantile_canonical"].isin(SELECTED_CANONICAL_QUANTILES)) &
        (summary_df["wave_case"].isin(available_wave_cases))
    ].copy()
    row_keys = []
    row_labels = []
    for wave_case in available_wave_cases:
        for q in SELECTED_CANONICAL_QUANTILES:
            row_keys.append((wave_case, q))
            row_labels.append(f"{wave_case_label(wave_case)} | {quantile_label_pretty(q)}")
    col_ids = [1, 2, 3]
    col_labels = [WETLAND_STYLE[c]["pretty"] for c in col_ids]
    mat_mean = np.full((len(row_keys), len(col_ids)), np.nan)
    mat_frac = np.full((len(row_keys), len(col_ids)), np.nan)
    for i, (wave_case, q) in enumerate(row_keys):
        for j, class_id in enumerate(col_ids):
            sel = sub[(sub["wave_case"] == wave_case) & (sub["quantile_canonical"] == q) & (sub["wetland_class_id"] == class_id)]
            if len(sel) > 0:
                mat_mean[i, j] = float(sel["delta_mean"].iloc[0])
                mat_frac[i, j] = 100.0 * float(sel["frac_linked_to_gt25_decrease"].iloc[0])
    fig, axes = plt.subplots(1, 2, figsize=(10.6, max(5.6, 0.42 * len(row_keys) + 2.2)))
    heatmap_panel(axes[0], mat_mean, row_labels, col_labels, cmap=make_pct_cmap(), vmin=-30, vmax=0, cbar_label=r"Mean $	au_{\mathrm{wave}}$ change (%)")
    annotate_panel(axes[0], "a")
    heatmap_panel(axes[1], mat_frac, row_labels, col_labels, cmap=make_frac_cmap(), vmin=0, vmax=max(5.0, safe_nanmax(mat_frac, fallback=5.0)), cbar_label=r"Wetlands linked to >25% $	au_{\mathrm{wave}}$ decrease (%)")
    annotate_panel(axes[1], "b")
    save_jpg(fig, FIGURE_DIR / "Sensitivity2_wave_quantile_tau.jpg")

def make_kernel_sensitivity_tau_figure(summary_df: pd.DataFrame) -> None:
    sub = summary_df[
        (summary_df["wave_case"] == MAIN_WAVE_CASE) &
        (summary_df["version"] == MAIN_VERSION) &
        (summary_df["scenario"] == MAIN_SCENARIO) &
        (summary_df["year"] == MAIN_YEAR) &
        (summary_df["quantile_canonical"] == MAIN_QUANTILE)
    ].copy()
    kernel_order = [k["label"] for k in KERNEL_SPECS]
    kernel_labels = [f"{int(k['lambda_km'])}/{int(k['dmax_km'])}" for k in KERNEL_SPECS]
    fig, axes = plt.subplots(1, 2, figsize=KERNEL_FIGSIZE)
    for class_id in [1, 2, 3]:
        ys_mean = []
        ys_frac = []
        for k in kernel_order:
            sel = sub[(sub["kernel_label"] == k) & (sub["wetland_class_id"] == class_id)]
            ys_mean.append(float(sel["delta_mean"].iloc[0]) if len(sel) > 0 else np.nan)
            ys_frac.append(100.0 * float(sel["frac_linked_to_gt25_decrease"].iloc[0]) if len(sel) > 0 else np.nan)
        axes[0].plot(np.arange(len(kernel_order)), ys_mean, marker="o", lw=1.6, color=WETLAND_STYLE[class_id]["color"], label=WETLAND_STYLE[class_id]["pretty"])
        axes[1].plot(np.arange(len(kernel_order)), ys_frac, marker="o", lw=1.6, color=WETLAND_STYLE[class_id]["color"], label=WETLAND_STYLE[class_id]["pretty"])
    axes[0].axhline(0, color="0.75", lw=0.8)
    axes[0].set_xticks(np.arange(len(kernel_order)))
    axes[0].set_xticklabels(kernel_labels)
    axes[0].set_xlabel(r"$\lambda$ / $D_{\max}$ (km)")
    axes[0].set_ylabel(r"Mean $	au_{\mathrm{wave}}$ change (%)")
    annotate_panel(axes[0], "a")
    axes[1].set_xticks(np.arange(len(kernel_order)))
    axes[1].set_xticklabels(kernel_labels)
    axes[1].set_xlabel(r"$\lambda$ / $D_{\max}$ (km)")
    axes[1].set_ylabel(r"Wetlands linked to >25% $	au_{\mathrm{wave}}$ decrease (%)")
    annotate_panel(axes[1], "b")
    axes[1].legend(frameon=False, loc="upper left")
    save_jpg(fig, FIGURE_DIR / "Sensitivity3_kernel_tau.jpg")

def make_scenario_sensitivity_severity_figure(summary_df: pd.DataFrame) -> None:
    sub = summary_df[
        (summary_df["kernel_label"] == MAIN_KERNEL_LABEL) &
        (summary_df["wave_case"] == MAIN_WAVE_CASE) &
        (summary_df["version"] == MAIN_VERSION) &
        (summary_df["quantile_canonical"] == MAIN_QUANTILE) &
        (summary_df["scenario"].isin(SELECTED_SCENARIOS)) &
        (summary_df["year"].isin(SELECTED_YEARS))
    ].copy()
    row_keys = []
    row_labels = []
    for year in SELECTED_YEARS:
        for scenario in SELECTED_SCENARIOS:
            row_keys.append((scenario, year))
            row_labels.append(f"{year} | {scenario_label(scenario)}")
    col_ids = [1, 2, 3]
    col_labels = [WETLAND_STYLE[c]["pretty"] for c in col_ids]
    mat_sev = np.full((len(row_keys), len(col_ids)), np.nan)
    for i, (scenario, year) in enumerate(row_keys):
        for j, class_id in enumerate(col_ids):
            sel = sub[(sub["scenario"] == scenario) & (sub["year"] == year) & (sub["wetland_class_id"] == class_id)]
            if len(sel) > 0:
                mat_sev[i, j] = float(sel["severity_rel_to_main_pct"].iloc[0])
    fig, ax = plt.subplots(1, 1, figsize=HEATMAP_FIGSIZE)
    heatmap_panel(ax, mat_sev, row_labels, col_labels, cmap=make_frac_cmap(), vmin=0, vmax=max(100.0, safe_nanmax(mat_sev, fallback=100.0)), cbar_label="Stress-weighted wetland area\n(relative to main case, %)")
    annotate_panel(ax, "a")
    save_jpg(fig, FIGURE_DIR / "Sensitivity4_scenario_year_severity.jpg")

def make_wave_quantile_sensitivity_severity_figure(summary_df: pd.DataFrame) -> None:
    available_wave_cases = [w for w in ANALYSIS_WAVE_CASES if w in set(summary_df["wave_case"].unique())]
    sub = summary_df[
        (summary_df["kernel_label"] == MAIN_KERNEL_LABEL) &
        (summary_df["version"] == MAIN_VERSION) &
        (summary_df["scenario"] == MAIN_SCENARIO) &
        (summary_df["year"] == MAIN_YEAR) &
        (summary_df["quantile_canonical"].isin(SELECTED_CANONICAL_QUANTILES)) &
        (summary_df["wave_case"].isin(available_wave_cases))
    ].copy()
    row_keys = []
    row_labels = []
    for wave_case in available_wave_cases:
        for q in SELECTED_CANONICAL_QUANTILES:
            row_keys.append((wave_case, q))
            row_labels.append(f"{wave_case_label(wave_case)} | {quantile_label_pretty(q)}")
    col_ids = [1, 2, 3]
    col_labels = [WETLAND_STYLE[c]["pretty"] for c in col_ids]
    mat_sev = np.full((len(row_keys), len(col_ids)), np.nan)
    for i, (wave_case, q) in enumerate(row_keys):
        for j, class_id in enumerate(col_ids):
            sel = sub[(sub["wave_case"] == wave_case) & (sub["quantile_canonical"] == q) & (sub["wetland_class_id"] == class_id)]
            if len(sel) > 0:
                mat_sev[i, j] = float(sel["severity_rel_to_main_pct"].iloc[0])
    fig, ax = plt.subplots(1, 1, figsize=(6.2, max(5.6, 0.42 * len(row_keys) + 2.0)))
    heatmap_panel(ax, mat_sev, row_labels, col_labels, cmap=make_frac_cmap(), vmin=0, vmax=max(100.0, safe_nanmax(mat_sev, fallback=100.0)), cbar_label="Stress-weighted wetland area\n(relative to main case, %)")
    annotate_panel(ax, "a")
    save_jpg(fig, FIGURE_DIR / "Sensitivity5_wave_quantile_severity.jpg")

def make_kernel_sensitivity_severity_figure(summary_df: pd.DataFrame) -> None:
    sub = summary_df[
        (summary_df["wave_case"] == MAIN_WAVE_CASE) &
        (summary_df["version"] == MAIN_VERSION) &
        (summary_df["scenario"] == MAIN_SCENARIO) &
        (summary_df["year"] == MAIN_YEAR) &
        (summary_df["quantile_canonical"] == MAIN_QUANTILE)
    ].copy()
    kernel_order = [k["label"] for k in KERNEL_SPECS]
    kernel_labels = [f"{int(k['lambda_km'])}/{int(k['dmax_km'])}" for k in KERNEL_SPECS]
    fig, ax = plt.subplots(1, 1, figsize=(5.4, 4.8))
    for class_id in [1, 2, 3]:
        ys = []
        for k in kernel_order:
            sel = sub[(sub["kernel_label"] == k) & (sub["wetland_class_id"] == class_id)]
            ys.append(float(sel["severity_rel_to_main_pct"].iloc[0]) if len(sel) > 0 else np.nan)
        ax.plot(np.arange(len(kernel_order)), ys, marker="o", lw=1.6, color=WETLAND_STYLE[class_id]["color"], label=WETLAND_STYLE[class_id]["pretty"])
    ax.axhline(100, color="0.75", lw=0.8, ls="--")
    ax.set_xticks(np.arange(len(kernel_order)))
    ax.set_xticklabels(kernel_labels)
    ax.set_xlabel(r"$\lambda$ / $D_{\max}$ (km)")
    ax.set_ylabel("Stress-weighted wetland area\n(relative to main case, %)")
    annotate_panel(ax, "a")
    ax.legend(frameon=False, loc="upper left")
    save_jpg(fig, FIGURE_DIR / "Sensitivity6_kernel_severity.jpg")


def extract_main_case_rows(summary_df: pd.DataFrame) -> pd.DataFrame:
    """Return the three wetland-class rows for the configured main case."""
    sub = summary_df[
        (summary_df["kernel_label"] == MAIN_KERNEL_LABEL)
        & (summary_df["wave_case"] == MAIN_WAVE_CASE)
        & (summary_df["version"] == MAIN_VERSION)
        & (summary_df["scenario"] == MAIN_SCENARIO)
        & (summary_df["year"] == MAIN_YEAR)
        & (summary_df["quantile_canonical"] == MAIN_QUANTILE)
    ].copy()
    sub = sub.sort_values("wetland_class_id").reset_index(drop=True)
    if sub["wetland_class_id"].astype(int).tolist() != [1, 2, 3]:
        raise RuntimeError("Main-case summary does not contain exactly wetland classes 1, 2, and 3.")
    return sub


def build_arctic_wetland_comparison(
    full_summary_df: pd.DataFrame,
    cutoff_summary_df: pd.DataFrame,
) -> pd.DataFrame:
    """Build a concise class-wise comparison of full versus <=66N results."""
    full = extract_main_case_rows(full_summary_df).set_index("wetland_class_id")
    cut = extract_main_case_rows(cutoff_summary_df).set_index("wetland_class_id")

    rows = []
    for class_id in [1, 2, 3]:
        f = full.loc[class_id]
        c = cut.loc[class_id]
        full_area = float(f["source_total_wetland_area_km2"])
        cut_area = float(c["source_total_wetland_area_km2"])
        full_severity = float(f["total_severity"])
        cut_severity = float(c["total_severity"])
        rows.append(
            {
                "wetland_class_id": class_id,
                "wetland_class": WETLAND_STYLE[class_id]["name"],
                "wetland_class_pretty": WETLAND_STYLE[class_id]["pretty"],
                "full_source_wetland_area_km2": full_area,
                "lat_le_66_source_wetland_area_km2": cut_area,
                "source_wetland_area_retained_pct": (
                    100.0 * cut_area / full_area if full_area > 0 else np.nan
                ),
                "full_delta_mean_pct": float(f["delta_mean"]),
                "lat_le_66_delta_mean_pct": float(c["delta_mean"]),
                "delta_mean_change_percentage_points": float(c["delta_mean"] - f["delta_mean"]),
                "full_frac_linked_gt25_pct": 100.0 * float(f["frac_linked_to_gt25_decrease"]),
                "lat_le_66_frac_linked_gt25_pct": 100.0 * float(c["frac_linked_to_gt25_decrease"]),
                "frac_linked_gt25_change_percentage_points": (
                    100.0 * float(c["frac_linked_to_gt25_decrease"] - f["frac_linked_to_gt25_decrease"])
                ),
                "full_total_severity": full_severity,
                "lat_le_66_total_severity": cut_severity,
                "total_severity_retained_pct": (
                    100.0 * cut_severity / full_severity
                    if np.isfinite(full_severity) and full_severity > 0 else np.nan
                ),
            }
        )
    return pd.DataFrame(rows)


def make_arctic_wetland_sensitivity_figure(comparison_df: pd.DataFrame) -> None:
    """Create the Nature-style 2x2 sensitivity figure for the 66N exclusion."""
    df = comparison_df.sort_values("wetland_class_id").reset_index(drop=True)
    class_ids = df["wetland_class_id"].astype(int).tolist()
    labels = [WETLAND_STYLE[c]["pretty"] for c in class_ids]
    colors = [WETLAND_STYLE[c]["color"] for c in class_ids]
    x = np.arange(len(class_ids), dtype=float)

    fig, axes = plt.subplots(2, 2, figsize=(10.2, 7.8))

    # a) Fraction of each wetland class retained after the latitude cutoff.
    vals = df["source_wetland_area_retained_pct"].to_numpy(dtype=float)
    axes[0, 0].bar(x, vals, color=colors, edgecolor="0.2", linewidth=0.6)
    axes[0, 0].axhline(100.0, color="0.65", lw=0.8, ls="--")
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(labels)
    axes[0, 0].set_ylabel("Source wetland area retained (%)")
    axes[0, 0].set_ylim(0, max(105.0, np.nanmax(vals) * 1.08))

    # b) Weighted mean tau-wave change.
    for i, class_id in enumerate(class_ids):
        y = [
            df.loc[i, "full_delta_mean_pct"],
            df.loc[i, "lat_le_66_delta_mean_pct"],
        ]
        axes[0, 1].plot(
            [0, 1],
            y,
            marker="o",
            lw=1.6,
            ms=5.0,
            color=WETLAND_STYLE[class_id]["color"],
            label=WETLAND_STYLE[class_id]["pretty"],
        )
    axes[0, 1].set_xticks([0, 1])
    axes[0, 1].set_xticklabels(["Full domain", r"$\leq 66^\circ$N"])
    axes[0, 1].set_ylabel(r"Weighted mean $\Delta\tau_{\mathrm{wave}}$ (%)")
    axes[0, 1].axhline(0.0, color="0.75", lw=0.8)
    axes[0, 1].legend(frameon=False, loc="best")

    # c) Fraction linked to a >25% decline.
    width = 0.34
    full_frac = df["full_frac_linked_gt25_pct"].to_numpy(dtype=float)
    cut_frac = df["lat_le_66_frac_linked_gt25_pct"].to_numpy(dtype=float)
    axes[1, 0].bar(
        x - width / 2,
        full_frac,
        width=width,
        facecolor="white",
        edgecolor=colors,
        linewidth=1.3,
        label="Full domain",
    )
    axes[1, 0].bar(
        x + width / 2,
        cut_frac,
        width=width,
        color=colors,
        edgecolor="0.2",
        linewidth=0.6,
        label=r"$\leq 66^\circ$N",
    )
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(labels)
    axes[1, 0].set_ylabel(
        r"Wetlands linked to >25% $\tau_{\mathrm{wave}}$ decrease (%)"
    )
    axes[1, 0].legend(frameon=False, loc="best")

    # d) Stress-weighted wetland area retained.
    sev_retained = df["total_severity_retained_pct"].to_numpy(dtype=float)
    axes[1, 1].bar(
        x,
        sev_retained,
        color=colors,
        edgecolor="0.2",
        linewidth=0.6,
    )
    axes[1, 1].axhline(100.0, color="0.65", lw=0.8, ls="--")
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(labels)
    axes[1, 1].set_ylabel("Stress-weighted wetland area\nretained (%)")
    axes[1, 1].set_ylim(0, max(105.0, np.nanmax(sev_retained) * 1.08))

    for ax, letter in zip(axes.ravel(), ["a", "b", "c", "d"]):
        annotate_panel(ax, letter)
        ax.tick_params(direction="out")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.subplots_adjust(
        left=0.09,
        right=0.985,
        bottom=0.09,
        top=0.97,
        wspace=0.30,
        hspace=0.34,
    )
    save_jpg(fig, ARCTIC_WETLAND_FIG)


def update_manuscript_summary_with_wetland(comparison_df: pd.DataFrame) -> None:
    """Append wetland robustness metrics to the compact manuscript table."""
    rows = []
    for rec in comparison_df.to_dict(orient="records"):
        group = rec["wetland_class_pretty"]
        metrics = [
            (
                "Source wetland area retained",
                100.0,
                rec["source_wetland_area_retained_pct"],
                "% of full domain",
            ),
            (
                "Weighted mean tau-wave change",
                rec["full_delta_mean_pct"],
                rec["lat_le_66_delta_mean_pct"],
                "%",
            ),
            (
                "Wetlands linked to >25% tau-wave decrease",
                rec["full_frac_linked_gt25_pct"],
                rec["lat_le_66_frac_linked_gt25_pct"],
                "%",
            ),
            (
                "Stress-weighted wetland area retained",
                100.0,
                rec["total_severity_retained_pct"],
                "% of full domain",
            ),
        ]
        for metric, full_value, cutoff_value, unit in metrics:
            full_value = float(full_value)
            cutoff_value = float(cutoff_value)
            difference = cutoff_value - full_value
            relative = (
                100.0 * difference / abs(full_value)
                if np.isfinite(full_value) and abs(full_value) > 0 else np.nan
            )
            rows.append(
                {
                    "analysis": "Wetland lifeline",
                    "group": group,
                    "metric": metric,
                    "full_domain_value": full_value,
                    "lat_le_66N_value": cutoff_value,
                    "difference": difference,
                    "relative_change_pct": relative,
                    "unit": unit,
                }
            )

    new_rows = pd.DataFrame(rows)
    if MANUSCRIPT_SUMMARY_CSV.exists():
        try:
            existing = pd.read_csv(MANUSCRIPT_SUMMARY_CSV)
            if "analysis" in existing.columns:
                existing = existing[existing["analysis"] != "Wetland lifeline"]
            combined = pd.concat([existing, new_rows], ignore_index=True)
        except Exception:
            combined = new_rows
    else:
        combined = new_rows

    combined.to_csv(MANUSCRIPT_SUMMARY_CSV, index=False)
    logging.info("Updated manuscript summary table: %s", MANUSCRIPT_SUMMARY_CSV)
    print("\nCompact Arctic-exclusion summary for manuscript use:")
    print(combined.to_string(index=False))


def export_main_case_rasters(
    offshore_points: OffshorePoints,
    kernel_load: KernelLoadResult,
    offshore_domain_mask: Optional[np.ndarray] = None,
) -> None:
    cid = main_case_id()
    case_idx = require_case_index(offshore_points, cid)
    delta = offshore_points.case_values[:, case_idx].astype(np.float64)
    negative = np.clip(-delta, 0, None)
    total_load = kernel_load.total_load.astype(np.float64)
    load_mg = kernel_load.class_loads[1].astype(np.float64)
    load_sm = kernel_load.class_loads[2].astype(np.float64)
    load_tf = kernel_load.class_loads[3].astype(np.float64)
    sev_mg = load_mg * negative
    sev_sm = load_sm * negative
    sev_tf = load_tf * negative
    sev_total = sev_mg + sev_sm + sev_tf

    if offshore_domain_mask is not None:
        domain = np.asarray(offshore_domain_mask, dtype=bool)
        delta = np.where(domain, delta, np.nan)
        load_mg = np.where(domain, load_mg, np.nan)
        load_sm = np.where(domain, load_sm, np.nan)
        load_tf = np.where(domain, load_tf, np.nan)
        total_load = np.where(domain, total_load, np.nan)
        sev_mg = np.where(domain, sev_mg, np.nan)
        sev_sm = np.where(domain, sev_sm, np.nan)
        sev_tf = np.where(domain, sev_tf, np.nan)
        sev_total = np.where(domain, sev_total, np.nan)

    write_geotiff(
        RASTER_DIR / "main_case_ocean_lifeline_loads.tif",
        [
            build_offshore_raster(load_mg, offshore_points),
            build_offshore_raster(load_sm, offshore_points),
            build_offshore_raster(load_tf, offshore_points),
            build_offshore_raster(total_load, offshore_points),
        ],
        offshore_points.grid,
        descriptions=[
            "Mangrove-linked ocean-side load",
            "Salt-marsh-linked ocean-side load",
            "Tidal-flat-linked ocean-side load",
            "Total wetland-linked ocean-side load",
        ],
    )
    write_geotiff(
        RASTER_DIR / "main_case_ocean_lifeline_severity.tif",
        [
            build_offshore_raster(sev_mg, offshore_points),
            build_offshore_raster(sev_sm, offshore_points),
            build_offshore_raster(sev_tf, offshore_points),
            build_offshore_raster(sev_total, offshore_points),
        ],
        offshore_points.grid,
        descriptions=["Mangrove severity", "Salt marsh severity", "Tidal flat severity", "Total severity"],
    )
    write_geotiff(
        RASTER_DIR / "main_case_tau_wave_change.tif",
        [build_offshore_raster(delta, offshore_points)],
        offshore_points.grid,
        descriptions=[r"Main-case tau_wave change (%)"],
    )

def main() -> None:
    logging.info("Starting FigureV1 ocean-side analysis.")
    logging.info("Output directory: %s", OUTPUT_ROOT)

    area_by_class_grid = aggregate_wetland_to_coarse_grid()
    wetland_points = build_wetland_points(area_by_class_grid)

    cases = discover_analysis_cases()
    offshore_points = build_offshore_points(cases)

    max_dmax_km = max(float(k["dmax_km"]) for k in KERNEL_SPECS)
    row, col, dist_km = load_or_build_connection_table(
        wetland_points=wetland_points,
        offshore_points=offshore_points,
        max_dmax_km=max_dmax_km,
    )

    # The global connection table is built once. The primary analysis then
    # applies <=66N masks to both offshore source cells and wetland receptor
    # cells at the pair level. This is equivalent to rebuilding the domain
    # after removing northern points, but avoids a second expensive KD-tree run.
    wet_domain_mask, off_domain_mask = latitude_domain_masks(
        wetland_points,
        offshore_points,
        MAX_ANALYSIS_LAT,
    )
    logging.info(
        "Primary latitude domain <= %.1fN: wetland cells retained %d/%d; offshore cells retained %d/%d",
        MAX_ANALYSIS_LAT,
        int(np.sum(wet_domain_mask)),
        len(wet_domain_mask),
        int(np.sum(off_domain_mask)),
        len(off_domain_mask),
    )

    kernel_results: Dict[str, KernelLoadResult] = {}
    all_summary: List[pd.DataFrame] = []

    for kernel_spec in KERNEL_SPECS:
        logging.info("Processing primary-domain kernel: %s", kernel_spec["label"])
        kernel_load = build_kernel_load_result(
            row=row,
            col=col,
            dist_km=dist_km,
            wetland_points=wetland_points,
            offshore_points=offshore_points,
            kernel_spec=kernel_spec,
            wetland_domain_mask=wet_domain_mask,
            offshore_domain_mask=off_domain_mask,
        )
        kernel_results[kernel_load.label] = kernel_load
        all_summary.append(
            compute_case_summary_for_kernel(
                kernel_load=kernel_load,
                offshore_points=offshore_points,
                wetland_points=wetland_points,
                wetland_domain_mask=wet_domain_mask,
            )
        )

    summary_df = pd.concat(all_summary, ignore_index=True)
    summary_df = compute_relative_severity(summary_df)

    summary_csv = TABLE_DIR / "ocean_side_lifeline_summary_all_cases.csv"
    summary_df.to_csv(summary_csv, index=False)
    logging.info("Saved primary-domain summary CSV: %s", summary_csv)

    # Full-domain benchmark for the Arctic-exclusion sensitivity test.
    full_kernel_spec = next(
        k for k in KERNEL_SPECS if str(k["label"]) == MAIN_KERNEL_LABEL
    )
    full_domain_kernel = build_kernel_load_result(
        row=row,
        col=col,
        dist_km=dist_km,
        wetland_points=wetland_points,
        offshore_points=offshore_points,
        kernel_spec=full_kernel_spec,
        wetland_domain_mask=None,
        offshore_domain_mask=None,
    )
    full_domain_summary_df = compute_case_summary_for_kernel(
        kernel_load=full_domain_kernel,
        offshore_points=offshore_points,
        wetland_points=wetland_points,
        wetland_domain_mask=None,
    )
    full_domain_summary_df = compute_relative_severity(full_domain_summary_df)
    full_domain_summary_df.to_csv(
        TABLE_DIR / "ocean_side_lifeline_summary_full_domain_main_kernel.csv",
        index=False,
    )

    arctic_comparison_df = build_arctic_wetland_comparison(
        full_domain_summary_df,
        summary_df,
    )
    arctic_comparison_df.to_csv(ARCTIC_WETLAND_SUMMARY_CSV, index=False)
    make_arctic_wetland_sensitivity_figure(arctic_comparison_df)
    update_manuscript_summary_with_wetland(arctic_comparison_df)

    try:
        with pd.ExcelWriter(TABLE_DIR / "ocean_side_lifeline_summary_all_cases.xlsx") as writer:
            summary_df.to_excel(writer, sheet_name="summary_all_cases", index=False)
            pd.DataFrame([c.__dict__ for c in cases]).to_excel(writer, sheet_name="analysis_cases", index=False)
            pd.DataFrame(KERNEL_SPECS).to_excel(writer, sheet_name="kernel_specs", index=False)
        logging.info("Saved summary Excel workbook.")
    except Exception as exc:
        logging.warning("Could not save Excel workbook. CSV is still available. Error: %s", exc)

    if MAIN_KERNEL_LABEL not in kernel_results:
        raise KeyError(f"Main kernel not found: {MAIN_KERNEL_LABEL}")

    export_main_case_rasters(
        offshore_points,
        kernel_results[MAIN_KERNEL_LABEL],
        offshore_domain_mask=off_domain_mask,
    )

    make_main_weakening_figure(offshore_points, kernel_results[MAIN_KERNEL_LABEL], summary_df)
    make_main_severity_figure(offshore_points, kernel_results[MAIN_KERNEL_LABEL], summary_df)

    make_scenario_sensitivity_tau_figure(summary_df)
    make_wave_quantile_sensitivity_tau_figure(summary_df)
    make_kernel_sensitivity_tau_figure(summary_df)

    make_scenario_sensitivity_severity_figure(summary_df)
    make_wave_quantile_sensitivity_severity_figure(summary_df)
    make_kernel_sensitivity_severity_figure(summary_df)

    save_json(
        METADATA_DIR / "run_metadata.json",
        {
            "wetland_path": str(WETLAND_PATH),
            "final_mask_path": str(FINAL_MASK_PATH),
            "shear_dir": str(SHEAR_DIR),
            "output_root": str(OUTPUT_ROOT),
            "run_label": RUN_LABEL,
            "coarse_res_deg": COARSE_RES_DEG,
            "max_analysis_lat_deg": MAX_ANALYSIS_LAT,
            "selected_versions": SELECTED_VERSIONS,
            "selected_scenarios": SELECTED_SCENARIOS,
            "selected_years": SELECTED_YEARS,
            "selected_quantiles": SELECTED_CANONICAL_QUANTILES,
            "analysis_wave_cases": ANALYSIS_WAVE_CASES,
            "kernel_specs": KERNEL_SPECS,
            "main_case": {
                "wave_case": MAIN_WAVE_CASE,
                "version": MAIN_VERSION,
                "scenario": MAIN_SCENARIO,
                "year": MAIN_YEAR,
                "quantile": MAIN_QUANTILE,
                "kernel_label": MAIN_KERNEL_LABEL,
            },
            "safe_numexpr_threads": SAFE_NUMEXPR_THREADS,
            "proj_data_dir": PROJ_DATA_DIR,
        },
    )

    logging.info("All analyses and figures completed successfully.")

if __name__ == "__main__":
    main()


2026-10-16 15:25:53,997 | INFO | SAFE_NUMEXPR_THREADS = 1
2026-10-16 15:25:53,998 | INFO | PROJ_DATA_DIR = /projectnb/nasa-marsh/Hangjie/conda/machine_learning/lib/python3.11/site-packages/pyproj/proj_dir/share/proj
2026-10-16 15:25:54,009 | INFO | Starting FigureV1 ocean-side analysis.
2026-10-16 15:25:54,009 | INFO | Output directory: /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures
2026-10-16 15:25:54,010 | INFO | Reusing existing wetland aggregation NPZ cache: /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/data/intermediate/coastal_wetland_area_by_class_0p1deg.npz
2026-10-16 15:25:54,182 | INFO | Coarse wetland receptor cells: 115565
2026-10-16 15:25:54,183 | INFO |   mangrove     area: 167307.02 km2 in 26727 coarse cells
2026-10-16 15:25:54,183 | INFO |   salt_marsh   area: 55267.65 km2 in 43394 coarse cells
2026-10-16 15:25:54,184 | INFO |   tidal_flat   area: 177839.33 km2 in 111405 coarse cells
2026-10-16 15:25:54,216 | INFO | Analysis cases discovered

Reading cases: 100%|████████████████████████████| 72/72 [00:03<00:00, 18.49it/s]

2026-10-16 15:25:58,192 | INFO | Offshore coastal-ring pixels in union mask: 4579
2026-10-16 15:25:58,193 | INFO | Reusing connection table: /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/data/intermediate/offshore_to_wetland_neighbors_coarse0p1_maxD300km.npz
2026-10-16 15:25:58,233 | INFO | Primary latitude domain <= 66.0N: wetland cells retained 80343/115565; offshore cells retained 3862/4579
2026-10-16 15:25:58,233 | INFO | Processing primary-domain kernel: lambda25_Dmax100


2026-10-16 15:25:58,332 | INFO | Processing primary-domain kernel: lambda50_Dmax150
2026-10-16 15:25:58,432 | INFO | Processing primary-domain kernel: lambda100_Dmax300
2026-10-16 15:25:58,556 | INFO | Saved primary-domain summary CSV: /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/data/tables/ocean_side_lifeline_summary_all_cases.csv
2026-10-16 15:25:59,159 | INFO | Saved figure: /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/FigureS_Arctic_exclusion_wetland_impacts.jpg
2026-10-16 15:25:59,164 | INFO | Updated manuscript summary table: /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/data/Arctic_exclusion_summary_for_manuscript.csv

Compact Arctic-exclusion summary for manuscript use:
        analysis      group                                    metric  full_domain_value  lat_le_66N_value    difference  relative_change_pct             unit
            WSCI     Global          Valid WSCI coastal area retained         100.000000         79.216168

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Rebuild Figure 6 from already-generated ocean-side wetland lifeline files.

This script does NOT rerun the expensive lifeline analysis.
It only reads previously generated outputs and creates:

    Figure6.jpg
    Figure6_panel_d_supplementary.jpg

Main Figure 6 design
--------------------
a) Global map:
   Declining tau_wave changes are shown with the main color scale.
   Increasing tau_wave changes are shown as dim triangular markers so that
   they remain visible but do not dominate the interpretation.

b) Weighted violin plot:
   Weighted distribution of tau_wave decreases for mangrove, salt marsh,
   and tidal flat. Increasing cells are excluded.

c) Stress-weighted wetland area (km^2):
   Linked wetland area weighted by the fractional stress decline:
       sum(linked area weight * max(-tau_wave change (%), 0) / 100)
   Increasing cells are excluded.

Supplementary figure design
---------------------------
The former Figure 6 panel d is moved to a standalone supplementary figure:

Exceedance curves:
   E_k(x) = P(Delta tau_wave <= -x | Wetland k, Delta tau_wave < 0)
   In practice, this is computed within the declining linked area only, so
   E_k(0) = 100% and the curve decreases toward 0% as the threshold increases.

Required existing files
-----------------------
1. tables/ocean_side_lifeline_summary_all_cases.csv
2. rasters/main_case_tau_wave_change.tif
3. rasters/main_case_ocean_lifeline_loads.tif

All comments are in English as requested.
"""

from __future__ import annotations

import math
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
import rasterio

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

try:
    import cartopy.crs as ccrs  # type: ignore
    import cartopy.feature as cfeature  # type: ignore
    HAS_CARTOPY = True
except Exception:
    HAS_CARTOPY = False


# =============================================================================
# User-adjustable typography
# =============================================================================

# Set to 1.00 for the original font sizes, or 1.25 for a 25% increase.
FONT_SCALE = 1.25


# =============================================================================
# User-adjustable paths
# =============================================================================

OUTPUT_ROOT = Path(
    "/projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures"
)
DATA_DIR = OUTPUT_ROOT / "data"

TABLE_DIR = DATA_DIR / "tables"
RASTER_DIR = DATA_DIR / "rasters"

SUMMARY_CSV = TABLE_DIR / "ocean_side_lifeline_summary_all_cases.csv"
TAU_RASTER = RASTER_DIR / "main_case_tau_wave_change.tif"

# Important: the correct filename has the final "s".
LOAD_RASTER = RASTER_DIR / "main_case_ocean_lifeline_loads.tif"

OUT_FIG = OUTPUT_ROOT / "Figure6.jpg"
OUT_SUPP_FIG = OUTPUT_ROOT / "Figure6_panel_d_supplementary.jpg"


# =============================================================================
# Main-case controls
# =============================================================================

MAIN_KERNEL_LABEL = "lambda50_Dmax150"
MAIN_WAVE_CASE = "combined_mean"
MAIN_VERSION = "withVLM"
MAIN_SCENARIO = "ssp585"
MAIN_YEAR = 2100
MAIN_QUANTILE = "q050"


# =============================================================================
# Figure layout controls
# =============================================================================

FIG_DPI = 520
JPG_QUALITY = 96

FONT_FAMILY = "serif"
BASE_FONT = 9.6 * FONT_SCALE
SMALL_FONT = 8.2 * FONT_SCALE
TINY_FONT = 7.2 * FONT_SCALE
PANEL_FONT = 12.0 * FONT_SCALE

FIG_SIZE = (9.8, 10.2)

# Top map panel.
PANEL_A_LEFT = 0.075
PANEL_A_RIGHT = 0.925
PANEL_A_BOTTOM = 0.455
PANEL_A_HEIGHT = 0.435
PANEL_A_WIDTH = PANEL_A_RIGHT - PANEL_A_LEFT

# Colorbar under map.
CBAR_WIDTH = 0.40
CBAR_LEFT = PANEL_A_LEFT + 0.5 * (PANEL_A_WIDTH - CBAR_WIDTH)
CBAR_BOTTOM = 0.410
CBAR_HEIGHT = 0.020

# Bottom row of the main figure: two equal-width panels.
# Panels b and c now occupy the full available width after panel d was moved
# to the Supplementary Information. Increase BOTTOM_GAP if more separation
# between panels b and c is desired.
BOTTOM_LEFT = 0.075
BOTTOM_RIGHT = 0.925
BOTTOM_WIDTH = BOTTOM_RIGHT - BOTTOM_LEFT
BOTTOM_GAP = 0.085
BOTTOM_BOTTOM = 0.080
BOTTOM_HEIGHT = 0.255

PANEL_B_WIDTH = (BOTTOM_WIDTH - BOTTOM_GAP) / 2.0
PANEL_C_WIDTH = PANEL_B_WIDTH

PANEL_B_LEFT = BOTTOM_LEFT
PANEL_C_LEFT = PANEL_B_LEFT + PANEL_B_WIDTH + BOTTOM_GAP

# Standalone supplementary figure containing the former panel d.
SUPP_FIG_SIZE = (5.2, 4.2)
SUPP_LEFT = 0.16
SUPP_BOTTOM = 0.17
SUPP_WIDTH = 0.80
SUPP_HEIGHT = 0.76

# Leave this empty for a single-panel supplementary figure.
# Set it to a label such as "a" later if this panel is merged into a
# multi-panel Supplementary Figure.
SUPP_PANEL_LABEL = ""


# =============================================================================
# Map controls
# =============================================================================

# Color scale for declining values only.
PCT_VMIN = -30.0
PCT_VMAX = 0.0
PCT_TICKS = [-30, -20, -10, 0]

MAP_ALPHA_DECLINE = 0.92

# Positive-change cells are displayed as dim triangular markers.
# These parameters are intentionally exposed here for easy manual adjustment.
MAP_POSITIVE_TRIANGLE_COLOR = "0.38"
MAP_POSITIVE_TRIANGLE_ALPHA = 0.32
MAP_POSITIVE_TRIANGLE_SIZE_FACTOR = 0.58
MAP_POSITIVE_TRIANGLE_MIN_SIZE = 1.0
MAP_POSITIVE_TRIANGLE_MAX_SIZE = 10.0

# Size legend position for linked wetland area.
SIZE_LEGEND_BBOX = (0.405, 0.020)
SIZE_LEGEND_LOC = "lower left"

# Positive-change legend position.
# Change POSITIVE_LEGEND_BBOX to manually move the legend.
POSITIVE_LEGEND_BBOX = (0.660, 0.100)
POSITIVE_LEGEND_LOC = "lower left"
POSITIVE_LEGEND_LABEL = r"$\tau_{\mathrm{wave}}$ increase"

MAP_SIZE_LEVEL_AREAS = (2.0, 10.0, 28.0)
MAP_SIZE_BIN_EDGE_QUANTILES = (45.0, 82.0)
MAP_SIZE_LEGEND_QUANTILES = (25.0, 62.0, 92.0)


# =============================================================================
# Violin and exceedance controls
# =============================================================================

VIOLIN_WIDTH = 0.36
VIOLIN_BINS = 220
VIOLIN_DENSE_POINTS = 700
VIOLIN_SMOOTH_SIGMA_BINS = 3.2
VIOLIN_RANGE_QUANTILES = (0.003, 0.997)
VIOLIN_TAIL_PAD_FRACTION = 0.10
VIOLIN_EDGE_COLOR = "0.18"
VIOLIN_EDGE_WIDTH = 0.55
VIOLIN_ALPHA = 0.96
SUMMARY_LINE_COLOR = "0.05"
SUMMARY_WHISKER_COLOR = "0.30"

EXCEEDANCE_MAX_THRESHOLD_AUTO = True
EXCEEDANCE_FIXED_MAX_THRESHOLD = 60.0
EXCEEDANCE_N_THRESHOLDS = 301
EXCEEDANCE_MIN_XMAX = 30.0
EXCEEDANCE_MAX_XMAX = 100.0


# =============================================================================
# Wetland style
# =============================================================================

WETLAND_STYLE = {
    1: {"name": "mangrove", "pretty": "Mangrove", "color": "#2b8c61"},
    2: {"name": "salt_marsh", "pretty": "Salt marsh", "color": "#c58c2b"},
    3: {"name": "tidal_flat", "pretty": "Tidal flat", "color": "#4c78a8"},
}

plt.rcParams.update(
    {
        "font.family": FONT_FAMILY,
        "font.size": BASE_FONT,
        "axes.labelsize": BASE_FONT,
        "xtick.labelsize": SMALL_FONT,
        "ytick.labelsize": SMALL_FONT,
        "legend.fontsize": SMALL_FONT,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)


# =============================================================================
# Basic utilities
# =============================================================================

def ensure_exists(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Required input file does not exist:\n{path}")

def save_jpg(fig: plt.Figure, path: Path, dpi: int = FIG_DPI) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=dpi, format="jpg", pil_kwargs={"quality": JPG_QUALITY})
    plt.close(fig)
    print(f"Saved figure: {path}")

def annotate_panel(ax, label: str, x: float = 0.0, y: float = 1.02) -> None:
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=PANEL_FONT,
        fontweight="bold",
        ha="left",
        va="bottom",
    )

def safe_nanmax(arr: np.ndarray, fallback: float = 1.0) -> float:
    arr = np.asarray(arr, dtype=float)
    if np.isfinite(arr).any():
        return float(np.nanmax(arr))
    return float(fallback)

def round_up_to_nearest(x: float, base: float = 10.0) -> float:
    if not np.isfinite(x) or x <= 0:
        return base
    return float(base * math.ceil(x / base))

def round_to_nice_number(x: float) -> float:
    if not np.isfinite(x) or x <= 0:
        return np.nan

    power = 10 ** math.floor(math.log10(x))
    base = x / power

    if base < 1.5:
        nice = 1.0
    elif base < 3.5:
        nice = 2.0
    elif base < 7.5:
        nice = 5.0
    else:
        nice = 10.0

    return nice * power

def format_load_label(v: float) -> str:
    if not np.isfinite(v):
        return "n/a"
    if v >= 1000:
        return f"{int(round(v)):,}"
    if v >= 10:
        return f"{int(round(v))}"
    return f"{v:.1f}"

def choose_scale(values: np.ndarray) -> Tuple[float, str]:
    vmax = safe_nanmax(np.abs(values), fallback=1.0)

    if vmax >= 1e9:
        return 1e9, r"$10^9$"
    if vmax >= 1e6:
        return 1e6, r"$10^6$"
    if vmax >= 1e3:
        return 1e3, r"$10^3$"

    return 1.0, ""


# =============================================================================
# Colormap
# =============================================================================

def make_pct_cmap() -> LinearSegmentedColormap:
    """
    Color map for negative tau_wave change.

    Dark blue means the largest decline.
    Pale near-zero colors mean weak decline.
    Positive changes are not passed into this color map; they are plotted
    separately as dim triangular markers.
    """
    colors = ["#0b2c6b", "#2b6cb0", "#6f73b8", "#c49a84", "#f7f2ea"]
    cmap = LinearSegmentedColormap.from_list("lifeline_weakening", colors, N=256)
    cmap.set_under("#081a4d")
    cmap.set_over("#f7f2ea")
    return cmap


# =============================================================================
# Map helpers
# =============================================================================

def draw_map_background(ax) -> None:
    if HAS_CARTOPY and hasattr(ax, "projection"):
        ax.set_global()
        try:
            ax.add_feature(
                cfeature.LAND.with_scale("110m"),
                facecolor="#f2efe9",
                edgecolor="none",
                zorder=0,
            )
        except Exception:
            pass
        try:
            ax.coastlines(resolution="110m", linewidth=0.35, color="0.35", zorder=3)
        except Exception:
            pass
    else:
        ax.set_xlim(-180, 180)
        ax.set_ylim(-90, 90)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.grid(False)

def scatter_declining_points(
    ax,
    lon: np.ndarray,
    lat: np.ndarray,
    values: np.ndarray,
    sizes: np.ndarray,
    cmap,
    norm,
):
    if HAS_CARTOPY and hasattr(ax, "projection"):
        return ax.scatter(
            lon, lat,
            c=values,
            s=sizes,
            cmap=cmap,
            norm=norm,
            marker="o",
            edgecolors="none",
            linewidths=0.0,
            alpha=MAP_ALPHA_DECLINE,
            transform=ccrs.PlateCarree(),
            rasterized=True,
            zorder=2.2,
        )

    return ax.scatter(
        lon, lat,
        c=values,
        s=sizes,
        cmap=cmap,
        norm=norm,
        marker="o",
        edgecolors="none",
        linewidths=0.0,
        alpha=MAP_ALPHA_DECLINE,
        rasterized=True,
        zorder=2.2,
    )

def scatter_increasing_triangles(
    ax,
    lon: np.ndarray,
    lat: np.ndarray,
    sizes: np.ndarray,
):
    """
    Plot positive tau_wave changes as dim triangles.

    These points are intentionally low contrast and smaller than declining
    points. They show that increases exist, while keeping the visual emphasis
    on the decline-dominated lifeline weakening signal.
    """
    triangle_sizes = np.clip(
        np.asarray(sizes, dtype=float) * MAP_POSITIVE_TRIANGLE_SIZE_FACTOR,
        MAP_POSITIVE_TRIANGLE_MIN_SIZE,
        MAP_POSITIVE_TRIANGLE_MAX_SIZE,
    )

    if HAS_CARTOPY and hasattr(ax, "projection"):
        return ax.scatter(
            lon,
            lat,
            s=triangle_sizes,
            marker="^",
            color=MAP_POSITIVE_TRIANGLE_COLOR,
            edgecolors="none",
            linewidths=0.0,
            alpha=MAP_POSITIVE_TRIANGLE_ALPHA,
            transform=ccrs.PlateCarree(),
            rasterized=True,
            zorder=1.7,
        )

    return ax.scatter(
        lon,
        lat,
        s=triangle_sizes,
        marker="^",
        color=MAP_POSITIVE_TRIANGLE_COLOR,
        edgecolors="none",
        linewidths=0.0,
        alpha=MAP_POSITIVE_TRIANGLE_ALPHA,
        rasterized=True,
        zorder=1.7,
    )

def marker_size_bin_edges(reference_load: np.ndarray) -> Tuple[float, float]:
    ref = np.asarray(reference_load, dtype=float)
    ok = np.isfinite(ref) & (ref > 0)

    if ok.sum() == 0:
        return 0.0, 1.0

    logv = np.log10(ref[ok] + 1.0)
    e1, e2 = np.nanpercentile(logv, MAP_SIZE_BIN_EDGE_QUANTILES)

    if not np.isfinite(e1) or not np.isfinite(e2) or e2 <= e1:
        e1, e2 = np.nanpercentile(logv, [33.0, 67.0])

    if not np.isfinite(e1) or not np.isfinite(e2) or e2 <= e1:
        e1, e2 = 0.0, 1.0

    return float(e1), float(e2)

def marker_size_class_index(load: np.ndarray, reference_load: np.ndarray | None = None) -> np.ndarray:
    load = np.asarray(load, dtype=float)
    ref = load if reference_load is None else np.asarray(reference_load, dtype=float)

    cls = np.full(load.shape, -1, dtype=np.int16)
    ok = np.isfinite(load) & (load > 0)

    if ok.sum() == 0:
        return cls

    e1, e2 = marker_size_bin_edges(ref)
    logv = np.log10(load[ok] + 1.0)

    cls_ok = np.zeros(logv.shape, dtype=np.int16)
    cls_ok[logv > e1] = 1
    cls_ok[logv > e2] = 2

    cls[ok] = cls_ok
    return cls

def marker_sizes_from_load(load: np.ndarray, reference_load: np.ndarray | None = None) -> np.ndarray:
    load = np.asarray(load, dtype=float)
    ref = load if reference_load is None else np.asarray(reference_load, dtype=float)

    sizes = np.full(load.shape, float(MAP_SIZE_LEVEL_AREAS[0]), dtype=float)
    cls = marker_size_class_index(load, ref)

    for i, area in enumerate(MAP_SIZE_LEVEL_AREAS):
        sizes[cls == i] = float(area)

    sizes[cls < 0] = float(MAP_SIZE_LEVEL_AREAS[0])
    return sizes

def size_legend_values(load: np.ndarray) -> Tuple[np.ndarray, List[str], np.ndarray]:
    load = np.asarray(load, dtype=float)
    pos = load[np.isfinite(load) & (load > 0)]

    if pos.size == 0:
        vals = np.array([1.0, 10.0, 100.0], dtype=float)
    else:
        vals = np.nanpercentile(pos, MAP_SIZE_LEGEND_QUANTILES).astype(float)
        rounded = np.array([round_to_nice_number(v) for v in vals], dtype=float)
        if np.all(np.isfinite(rounded)) and np.all(np.diff(rounded) > 0):
            vals = rounded

    vals = np.clip(vals, 1e-6, None)
    labels = [format_load_label(v) for v in vals]
    sizes = np.asarray(MAP_SIZE_LEVEL_AREAS, dtype=float)

    return vals, labels, sizes

def add_size_legend(ax, load: np.ndarray):
    _, labels, sizes = size_legend_values(load)

    handles = [
        Line2D(
            [0], [0],
            marker="o",
            linestyle="None",
            markerfacecolor="0.55",
            markeredgecolor="0.30",
            markeredgewidth=0.35,
            markersize=math.sqrt(float(s)),
            label=lab,
            alpha=0.95,
        )
        for s, lab in zip(sizes, labels)
    ]

    leg = ax.legend(
        handles=handles,
        title=r"Linked wetland area (km$^2$)",
        loc=SIZE_LEGEND_LOC,
        bbox_to_anchor=SIZE_LEGEND_BBOX,
        frameon=True,
        fancybox=False,
        borderpad=0.35,
        handletextpad=0.9,
        labelspacing=0.55,
        borderaxespad=0.0,
    )

    leg.get_frame().set_linewidth(0.5)
    leg.get_frame().set_alpha(0.95)

    if leg.get_title() is not None:
        leg.get_title().set_fontsize(SMALL_FONT)

    return leg

def add_positive_triangle_legend(ax):
    """
    Add a separate legend for positive-change triangular markers.

    Because matplotlib keeps only one active legend by default, the size legend
    is first added back as an artist before this legend is created.
    """
    handle = Line2D(
        [0], [0],
        marker="^",
        linestyle="None",
        markerfacecolor=MAP_POSITIVE_TRIANGLE_COLOR,
        markeredgecolor="none",
        markersize=5.4,
        alpha=MAP_POSITIVE_TRIANGLE_ALPHA,
        label=POSITIVE_LEGEND_LABEL,
    )

    leg = ax.legend(
        handles=[handle],
        loc=POSITIVE_LEGEND_LOC,
        bbox_to_anchor=POSITIVE_LEGEND_BBOX,
        frameon=True,
        fancybox=False,
        borderpad=0.35,
        handletextpad=0.65,
        labelspacing=0.45,
        borderaxespad=0.0,
    )

    leg.get_frame().set_linewidth(0.5)
    leg.get_frame().set_alpha(0.92)

    return leg


# =============================================================================
# Weighted statistics and violin helpers
# =============================================================================

def weighted_quantile(values: np.ndarray, weights: np.ndarray, probs: Iterable[float]) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    probs = np.asarray(list(probs), dtype=np.float64)

    ok = np.isfinite(values) & np.isfinite(weights) & (weights > 0)

    if ok.sum() == 0:
        return np.full(probs.shape, np.nan, dtype=np.float64)

    v = values[ok]
    w = weights[ok]

    order = np.argsort(v)
    v = v[order]
    w = w[order]

    cw = np.cumsum(w)
    total = cw[-1]

    if total <= 0:
        return np.full(probs.shape, np.nan, dtype=np.float64)

    q = (cw - 0.5 * w) / total
    return np.interp(probs, q, v)

def gaussian_smooth_1d(y: np.ndarray, sigma_bins: float) -> np.ndarray:
    sigma_bins = float(max(0.5, sigma_bins))
    radius = int(max(3, math.ceil(sigma_bins * 4)))

    x = np.arange(-radius, radius + 1)
    k = np.exp(-0.5 * (x / sigma_bins) ** 2)
    k /= np.sum(k)

    return np.convolve(y, k, mode="same")

def draw_weighted_violin(ax, xpos: float, values: np.ndarray, weights: np.ndarray, color: str) -> None:
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)

    # Panel b intentionally excludes increasing cells.
    ok = np.isfinite(values) & np.isfinite(weights) & (weights > 0) & (values < 0)

    if ok.sum() < 5:
        return

    v = values[ok]
    w = weights[ok]

    qlo, qhi = weighted_quantile(v, w, VIOLIN_RANGE_QUANTILES)

    if not np.isfinite(qlo) or not np.isfinite(qhi) or qhi <= qlo:
        qlo = float(np.nanmin(v))
        qhi = float(np.nanmax(v))

    vrange = max(qhi - qlo, 1e-6)
    pad = vrange * VIOLIN_TAIL_PAD_FRACTION
    ymin = qlo - pad
    ymax = qhi + pad

    edges = np.linspace(ymin, ymax, VIOLIN_BINS + 1)
    hist, _ = np.histogram(v, bins=edges, weights=w)
    centers = 0.5 * (edges[:-1] + edges[1:])

    density = hist.astype(np.float64)

    if np.sum(density) <= 0:
        return

    density = gaussian_smooth_1d(density, VIOLIN_SMOOTH_SIGMA_BINS)
    density = np.clip(density, 0, None)

    dense_y = np.linspace(centers[0], centers[-1], VIOLIN_DENSE_POINTS)
    dense_d = np.interp(dense_y, centers, density)

    if np.nanmax(dense_d) > 0:
        dense_d = dense_d / np.nanmax(dense_d)

    half_width = VIOLIN_WIDTH * dense_d
    left = xpos - half_width
    right = xpos + half_width

    ax.fill_betweenx(
        dense_y,
        left,
        right,
        facecolor=color,
        edgecolor=VIOLIN_EDGE_COLOR,
        linewidth=VIOLIN_EDGE_WIDTH,
        alpha=VIOLIN_ALPHA,
        zorder=2,
    )

    q25, q50, q75 = weighted_quantile(v, w, [0.25, 0.50, 0.75])

    ax.plot([xpos, xpos], [q25, q75], color=SUMMARY_WHISKER_COLOR, lw=1.3, zorder=4)
    ax.plot([xpos - 0.14, xpos + 0.14], [q50, q50], color=SUMMARY_LINE_COLOR, lw=1.8, zorder=5)


# =============================================================================
# Raster/table readers
# =============================================================================

def read_main_case_point_data(tau_raster: Path, load_raster: Path) -> Dict[str, np.ndarray]:
    """
    Read already-generated main-case rasters and convert valid grid cells to point arrays.

    LOAD_RASTER bands are expected to be:
        1 = mangrove-linked load
        2 = salt-marsh-linked load
        3 = tidal-flat-linked load
        4 = total linked load

    TAU_RASTER band is expected to be:
        1 = tau_wave change (%)
    """
    with rasterio.open(tau_raster) as src_tau, rasterio.open(load_raster) as src_load:
        if (src_tau.width != src_load.width) or (src_tau.height != src_load.height):
            raise RuntimeError("Tau raster and load raster do not have the same shape.")

        if src_tau.transform != src_load.transform:
            raise RuntimeError("Tau raster and load raster do not share the same grid transform.")

        tau = src_tau.read(1).astype(np.float32)
        tau_nodata = src_tau.nodata
        if tau_nodata is not None:
            tau[np.isclose(tau, tau_nodata)] = np.nan

        load_mg = src_load.read(1).astype(np.float32)
        load_sm = src_load.read(2).astype(np.float32)
        load_tf = src_load.read(3).astype(np.float32)
        load_total = src_load.read(4).astype(np.float32)

        load_nodata = src_load.nodata
        if load_nodata is not None:
            for arr in [load_mg, load_sm, load_tf, load_total]:
                arr[np.isclose(arr, load_nodata)] = np.nan

        transform = src_tau.transform
        bounds = src_tau.bounds
        width = src_tau.width
        height = src_tau.height

    lon = bounds.left + (np.arange(width, dtype=np.float64) + 0.5) * transform.a
    lat = bounds.top + (np.arange(height, dtype=np.float64) + 0.5) * transform.e
    lon2, lat2 = np.meshgrid(lon, lat)

    valid = np.isfinite(tau) & np.isfinite(load_total) & (load_total > 0)

    return {
        "lon": lon2[valid].astype(np.float64),
        "lat": lat2[valid].astype(np.float64),
        "delta": tau[valid].astype(np.float64),
        "load_mg": load_mg[valid].astype(np.float64),
        "load_sm": load_sm[valid].astype(np.float64),
        "load_tf": load_tf[valid].astype(np.float64),
        "load_total": load_total[valid].astype(np.float64),
    }

def read_main_case_summary(summary_csv: Path) -> pd.DataFrame:
    df = pd.read_csv(summary_csv)

    mask = (
        (df["kernel_label"] == MAIN_KERNEL_LABEL) &
        (df["wave_case"] == MAIN_WAVE_CASE) &
        (df["version"] == MAIN_VERSION) &
        (df["scenario"] == MAIN_SCENARIO) &
        (df["year"] == MAIN_YEAR) &
        (df["quantile_canonical"] == MAIN_QUANTILE)
    )

    sub = df.loc[mask].copy()

    if sub.empty:
        raise RuntimeError(
            "No rows found in summary CSV for the requested main case:\n"
            f"kernel={MAIN_KERNEL_LABEL}, wave_case={MAIN_WAVE_CASE}, version={MAIN_VERSION}, "
            f"scenario={MAIN_SCENARIO}, year={MAIN_YEAR}, quantile={MAIN_QUANTILE}"
        )

    sub = sub.sort_values("wetland_class_id").reset_index(drop=True)

    expected = [1, 2, 3]
    found = sorted(sub["wetland_class_id"].astype(int).unique().tolist())

    if found != expected:
        raise RuntimeError(f"Expected wetland_class_id = {expected}, but got {found}")

    return sub


# =============================================================================
# Stress-weighted wetland area and exceedance calculations
# =============================================================================

def compute_cumulative_severity_from_points(
    delta: np.ndarray,
    class_loads: Dict[int, np.ndarray],
) -> Dict[int, float]:
    """
    Compute stress-weighted wetland area (km^2) from point-level data.

    The legacy function name is retained for compatibility.
    Delta tau is stored as a percentage (30 for 30%); divide by 100
    to weight linked wetland area by the dimensionless fractional loss:
        area_k = sum_i load_{i,k} * max(-Delta tau_i, 0) / 100

    Increasing cells contribute zero.
    """
    out: Dict[int, float] = {}
    negative_magnitude = np.where(np.isfinite(delta) & (delta < 0), -delta, 0.0) / 100.0

    for class_id in [1, 2, 3]:
        load = np.asarray(class_loads[class_id], dtype=float)
        ok = np.isfinite(load) & (load > 0) & np.isfinite(negative_magnitude)
        out[class_id] = float(np.sum(load[ok] * negative_magnitude[ok])) if ok.any() else np.nan

    return out

def determine_exceedance_xmax(delta: np.ndarray, class_loads: Dict[int, np.ndarray]) -> float:
    """
    Choose a readable x-axis maximum for exceedance curves.

    The upper bound is based on the 99.5th percentile of all decline magnitudes
    weighted by the presence of any linked wetland load.
    """
    if not EXCEEDANCE_MAX_THRESHOLD_AUTO:
        return float(EXCEEDANCE_FIXED_MAX_THRESHOLD)

    total_load = np.zeros_like(delta, dtype=float)
    for class_id in [1, 2, 3]:
        load = np.asarray(class_loads[class_id], dtype=float)
        total_load += np.where(np.isfinite(load), load, 0.0)

    ok = np.isfinite(delta) & (delta < 0) & np.isfinite(total_load) & (total_load > 0)

    if ok.sum() == 0:
        return float(EXCEEDANCE_FIXED_MAX_THRESHOLD)

    decline_mag = -delta[ok]
    x99 = float(np.nanpercentile(decline_mag, 99.5))
    xmax = round_up_to_nearest(x99, base=10.0)
    xmax = max(EXCEEDANCE_MIN_XMAX, min(EXCEEDANCE_MAX_XMAX, xmax))

    return float(xmax)

def compute_exceedance_curve(
    delta: np.ndarray,
    weights: np.ndarray,
    thresholds: np.ndarray,
) -> np.ndarray:
    """
    Compute the exceedance curve within the declining linked area only.

    E_k(x) = P(Delta tau_wave <= -x | Wetland k, Delta tau_wave < 0)

    This denominator choice follows the plotting decision that panels b/c/d
    should ignore increasing cells. Therefore E_k(0) is exactly 100% when
    there are valid declining linked cells.
    """
    delta = np.asarray(delta, dtype=float)
    weights = np.asarray(weights, dtype=float)
    thresholds = np.asarray(thresholds, dtype=float)

    ok = np.isfinite(delta) & (delta < 0) & np.isfinite(weights) & (weights > 0)

    if ok.sum() == 0:
        return np.full(thresholds.shape, np.nan, dtype=float)

    loss = -delta[ok]
    w = weights[ok]
    total = float(np.sum(w))

    if total <= 0:
        return np.full(thresholds.shape, np.nan, dtype=float)

    exceed = np.empty(thresholds.shape, dtype=float)

    for i, x in enumerate(thresholds):
        exceed[i] = 100.0 * float(np.sum(w[loss >= x]) / total)

    return exceed


# =============================================================================
# Main plotting function
# =============================================================================

def make_figure6() -> None:
    ensure_exists(SUMMARY_CSV)
    ensure_exists(TAU_RASTER)
    ensure_exists(LOAD_RASTER)

    point_data = read_main_case_point_data(TAU_RASTER, LOAD_RASTER)

    # The summary table is read for validation and consistency with the original
    # workflow. Panel c is recomputed from point-level rasters so that increasing
    # cells are explicitly ignored in this plotting script.
    _summary = read_main_case_summary(SUMMARY_CSV)

    lon = point_data["lon"]
    lat = point_data["lat"]
    delta = point_data["delta"]
    load_total = point_data["load_total"]

    class_loads = {
        1: point_data["load_mg"],
        2: point_data["load_sm"],
        3: point_data["load_tf"],
    }

    valid_all = np.isfinite(delta) & np.isfinite(load_total) & (load_total > 0)
    decline_map = valid_all & (delta <= 0)
    increase_map = valid_all & (delta > 0)

    fig = plt.figure(figsize=FIG_SIZE)

    # -------------------------------------------------------------------------
    # Panel a: global map
    # -------------------------------------------------------------------------
    if HAS_CARTOPY:
        axa = fig.add_axes(
            [PANEL_A_LEFT, PANEL_A_BOTTOM, PANEL_A_WIDTH, PANEL_A_HEIGHT],
            projection=ccrs.Robinson(),
        )
    else:
        axa = fig.add_axes([PANEL_A_LEFT, PANEL_A_BOTTOM, PANEL_A_WIDTH, PANEL_A_HEIGHT])

    draw_map_background(axa)

    cmap = make_pct_cmap()
    norm = Normalize(vmin=PCT_VMIN, vmax=PCT_VMAX)

    sizes_all = marker_sizes_from_load(load_total[valid_all], reference_load=load_total[valid_all])
    sizes_full = np.full(delta.shape, float(MAP_SIZE_LEVEL_AREAS[0]), dtype=float)
    sizes_full[valid_all] = sizes_all

    # Plot increases first and quietly as dim triangles.
    if np.any(increase_map):
        scatter_increasing_triangles(
            axa,
            lon[increase_map],
            lat[increase_map],
            sizes_full[increase_map],
        )

    # Plot declines with the main color scale.
    if np.any(decline_map):
        scatter_declining_points(
            axa,
            lon[decline_map],
            lat[decline_map],
            np.clip(delta[decline_map], PCT_VMIN, PCT_VMAX),
            sizes_full[decline_map],
            cmap=cmap,
            norm=norm,
        )

    annotate_panel(axa, "a")

    size_legend = add_size_legend(axa, load_total[valid_all])
    axa.add_artist(size_legend)
    add_positive_triangle_legend(axa)

    cax = fig.add_axes([CBAR_LEFT, CBAR_BOTTOM, CBAR_WIDTH, CBAR_HEIGHT])
    cb = plt.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=cax, orientation="horizontal")
    cb.set_label(r"$\tau_{\mathrm{wave}}$ change (%)")
    cb.set_ticks(PCT_TICKS)

    # -------------------------------------------------------------------------
    # Panel b: weighted violin for declining cells only
    # -------------------------------------------------------------------------
    axb = fig.add_axes([PANEL_B_LEFT, BOTTOM_BOTTOM, PANEL_B_WIDTH, BOTTOM_HEIGHT])

    for xpos, class_id in enumerate([1, 2, 3], start=1):
        draw_weighted_violin(
            axb,
            xpos,
            delta,
            class_loads[class_id],
            color=WETLAND_STYLE[class_id]["color"],
        )

    axb.axhline(0, color="0.75", lw=0.8, zorder=0)
    axb.set_xlim(0.45, 3.55)
    axb.set_xticks([1, 2, 3])
    axb.set_xticklabels([WETLAND_STYLE[c]["pretty"] for c in [1, 2, 3]])
    axb.set_ylabel(r"$\tau_{\mathrm{wave}}$ change (%)")
    # axb.set_title("Declining linked area", fontsize=SMALL_FONT, pad=4.0)
    annotate_panel(axb, "b")

    # -------------------------------------------------------------------------
    # Panel c: stress-weighted wetland area (km^2)
    # -------------------------------------------------------------------------
    axc = fig.add_axes([PANEL_C_LEFT, BOTTOM_BOTTOM, PANEL_C_WIDTH, BOTTOM_HEIGHT])

    severity_dict = compute_cumulative_severity_from_points(delta, class_loads)
    severity_vals = np.array([severity_dict[c] for c in [1, 2, 3]], dtype=float)

    x = np.arange(3)
    colors = [WETLAND_STYLE[c]["color"] for c in [1, 2, 3]]

    scale, scale_txt = choose_scale(severity_vals)
    sev_plot = severity_vals / scale

    bars = axc.bar(
        x,
        sev_plot,
        color=colors,
        edgecolor="0.20",
        linewidth=0.55,
        width=0.70,
    )

    axc.set_xticks(x)
    axc.set_xticklabels([WETLAND_STYLE[c]["pretty"] for c in [1, 2, 3]])

    # Use fractional stress loss (25% = 0.25), so the result is in km^2.
    # The 10^n multiplier, when present, only scales the tick values.
    if scale_txt:
        axc.set_ylabel(
            "Stress-weighted wetland area\n" + rf"({scale_txt} km$^2$)"
        )
    else:
        axc.set_ylabel("Stress-weighted wetland area\n" + r"(km$^2$)")

    annotate_panel(axc, "c")

    ymax_c = safe_nanmax(sev_plot, fallback=1.0)
    axc.set_ylim(0, ymax_c * 1.18)

    for rect, raw_val, plot_val in zip(bars, severity_vals, sev_plot):
        if np.isfinite(plot_val):
            axc.text(
                rect.get_x() + rect.get_width() / 2.0,
                rect.get_height() + ymax_c * 0.03,
                f"{raw_val:.2e}",
                ha="center",
                va="bottom",
                fontsize=TINY_FONT,
            )

    # Save the reorganized main Figure 6 with panels a-c only.
    save_jpg(fig, OUT_FIG)

    # -------------------------------------------------------------------------
    # Supplementary figure: former Figure 6 panel d (exceedance curves)
    # -------------------------------------------------------------------------
    fig_supp = plt.figure(figsize=SUPP_FIG_SIZE)
    axs = fig_supp.add_axes([SUPP_LEFT, SUPP_BOTTOM, SUPP_WIDTH, SUPP_HEIGHT])

    xmax = determine_exceedance_xmax(delta, class_loads)
    thresholds = np.linspace(0.0, xmax, EXCEEDANCE_N_THRESHOLDS)

    for class_id in [1, 2, 3]:
        curve = compute_exceedance_curve(
            delta=delta,
            weights=class_loads[class_id],
            thresholds=thresholds,
        )

        axs.plot(
            thresholds,
            curve,
            color=WETLAND_STYLE[class_id]["color"],
            lw=1.9,
            label=WETLAND_STYLE[class_id]["pretty"],
        )

    axs.set_xlim(0, xmax)
    axs.set_ylim(0, 100)
    axs.set_xlabel(r"$\tau_{\mathrm{wave}}$ decrease threshold, $x$ (%)")
    axs.set_ylabel(r"Area exceeding threshold (%)")
    # axs.set_title("Exceedance of severe drops", fontsize=SMALL_FONT, pad=4.0)
    axs.grid(axis="both", color="0.88", lw=0.6)
    axs.legend(frameon=False, loc="upper right", handlelength=1.5)

    if SUPP_PANEL_LABEL:
        annotate_panel(axs, SUPP_PANEL_LABEL)

    save_jpg(fig_supp, OUT_SUPP_FIG)


# =============================================================================
# Entrypoint
# =============================================================================

def main() -> None:
    print("Rebuilding Figure 6 from existing outputs only.")
    print(f"Summary CSV : {SUMMARY_CSV}")
    print(f"Tau raster  : {TAU_RASTER}")
    print(f"Load raster : {LOAD_RASTER}")
    print(f"Main output : {OUT_FIG}")
    print(f"Supp output : {OUT_SUPP_FIG}")

    make_figure6()

    print("Done.")


if __name__ == "__main__":
    main()


Rebuilding Figure 6 from existing outputs only.
Summary CSV : /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/data/tables/ocean_side_lifeline_summary_all_cases.csv
Tau raster  : /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/data/rasters/main_case_tau_wave_change.tif
Load raster : /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/data/rasters/main_case_ocean_lifeline_loads.tif
Main output : /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/Figure6.jpg
Supp output : /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/Figure6_panel_d_supplementary.jpg
Saved figure: /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/Figure6.jpg
Saved figure: /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/Figure6_panel_d_supplementary.jpg
Done.


In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Replot selected sensitivity figures for the global ocean-side wetland lifeline analysis.

This script does two things:
1. Create a new 1x3 scenario/year sensitivity figure by combining:
   - Sensitivity1_scenario_year_tau (panel a and panel b)
   - Sensitivity4_scenario_year_severity (panel c)
   Each panel has its own horizontal colorbar placed directly below the panel.
   Panel a uses its own colormap, while panels b and c share the same colormap.
   All three panels use the same row order, and only the left panel displays
   the y-axis labels.
2. Regenerate Sensitivity3_kernel_tau.jpg with updated y-axis limits and legend position.

The script only reads the already-generated summary table and does not rerun the main analysis.
All comments are in English as requested.
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =============================================================================
# User-adjustable paths
# =============================================================================

OUTPUT_ROOT = Path(
    "/projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures"
)
DATA_DIR = OUTPUT_ROOT / "data"
TABLE_DIR = DATA_DIR / "tables"
FIGURE_DIR = OUTPUT_ROOT
SUMMARY_CSV = TABLE_DIR / "ocean_side_lifeline_summary_all_cases.csv"

# =============================================================================
# User-adjustable analysis controls
# =============================================================================

SELECTED_SCENARIOS = ["ssp126", "ssp245", "ssp585"]
SELECTED_YEARS = [2050, 2100]

MAIN_WAVE_CASE = "combined_mean"
MAIN_VERSION = "withVLM"
MAIN_SCENARIO = "ssp585"
MAIN_YEAR = 2100
MAIN_QUANTILE = "q050"
MAIN_KERNEL_LABEL = "lambda50_Dmax150"

KERNEL_SPECS = [
    {"lambda_km": 25.0, "dmax_km": 100.0, "label": "lambda25_Dmax100"},
    {"lambda_km": 50.0, "dmax_km": 150.0, "label": "lambda50_Dmax150"},
    {"lambda_km": 100.0, "dmax_km": 300.0, "label": "lambda100_Dmax300"},
]

# =============================================================================
# User-adjustable figure controls
# =============================================================================

FIG_DPI = 520
JPG_QUALITY = 96

FONT_FAMILY = "serif"
BASE_FONT = 9.2
SMALL_FONT = 8.0
TINY_FONT = 7.0
PANEL_FONT = 12.0

# New 1x3 figure controls
SCENARIO_1X3_FIGSIZE = (14.3, 5.6)
SCENARIO_1X3_WSPACE = 0.10
SCENARIO_1X3_HSPACE = 0.16
SCENARIO_1X3_CBAR_HEIGHT_RATIO = 0.09

# Kernel sensitivity figure controls
KERNEL_FIGSIZE = (9.2, 4.8)
KERNEL_LINEWIDTH = 1.8
KERNEL_MARKER_SIZE = 5.0
KERNEL_PANEL_A_YMAX = -15.0

# Panel b is a positive percentage metric, so the display limit is set to +50.
# If you truly want to force it to -50, change this value manually.
KERNEL_PANEL_B_YMAX = 50.0

KERNEL_LEGEND_LOC = "upper right"

# Panel a uses a dedicated colormap
PANEL_A_CMAP = LinearSegmentedColormap.from_list(
    "tau_change_cmap",
    ["#0b2c6b", "#2b6cb0", "#6f73b8", "#c49a84", "#f7f2ea"],
    N=256,
)

# Panels b and c share the same colormap
COMMON_HEATMAP_CMAP = LinearSegmentedColormap.from_list(
    "common_heatmap",
    ["#f7fbff", "#c6dbef", "#6baed6", "#2171b5", "#08306b"],
    N=256,
)

# Value ranges for the three heatmap panels
TAU_MEAN_VMIN = -30.0
TAU_MEAN_VMAX = 0.0
TAU_FRACTION_VMIN = 0.0
SEVERITY_VMIN = 0.0

WETLAND_STYLE: Dict[int, Dict[str, str]] = {
    1: {"name": "mangrove", "pretty": "Mangrove", "color": "#2b8c61"},
    2: {"name": "salt_marsh", "pretty": "Salt marsh", "color": "#c58c2b"},
    3: {"name": "tidal_flat", "pretty": "Tidal flat", "color": "#4c78a8"},
}

plt.rcParams.update(
    {
        "font.family": FONT_FAMILY,
        "font.size": BASE_FONT,
        "axes.labelsize": BASE_FONT,
        "xtick.labelsize": SMALL_FONT,
        "ytick.labelsize": SMALL_FONT,
        "legend.fontsize": SMALL_FONT,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

# =============================================================================
# Helper functions
# =============================================================================

def ensure_dir(path: Path) -> None:
    """Create a directory if it does not exist."""
    path.mkdir(parents=True, exist_ok=True)


def save_jpg(fig: plt.Figure, path: Path, dpi: int = FIG_DPI) -> None:
    """Save a figure as JPG with consistent output settings."""
    ensure_dir(path.parent)
    fig.savefig(path, dpi=dpi, format="jpg", pil_kwargs={"quality": JPG_QUALITY})
    plt.close(fig)
    print(f"Saved figure: {path}")


def scenario_label(s: str) -> str:
    """Convert scenario code into a prettier label."""
    mapping = {
        "ssp119": "SSP1-1.9",
        "ssp126": "SSP1-2.6",
        "ssp245": "SSP2-4.5",
        "ssp370": "SSP3-7.0",
        "ssp585": "SSP5-8.5",
    }
    return mapping.get(str(s).lower(), str(s).upper())


def safe_nanmax(arr: np.ndarray, fallback: float = 1.0) -> float:
    """Return nanmax safely, or a fallback if all values are invalid."""
    arr = np.asarray(arr, dtype=float)
    ok = np.isfinite(arr)
    if ok.sum() == 0:
        return fallback
    return float(np.nanmax(arr[ok]))


def annotate_panel(ax, text: str, x: float = -0.07, y: float = 0.95) -> None:
    """Place a bold panel label on an axes."""
    ax.text(
        x,
        y,
        text,
        transform=ax.transAxes,
        fontsize=PANEL_FONT,
        fontweight="bold",
        ha="left",
        va="bottom",
    )


def build_scenario_year_matrices(
    summary_df: pd.DataFrame,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str], List[str]]:
    """
    Build three matrices for the new 1x3 figure:
    panel a: mean tau change
    panel b: fraction linked to >25% tau decrease
    panel c: stress-weighted wetland area relative to main case
    """
    sub = summary_df[
        (summary_df["kernel_label"] == MAIN_KERNEL_LABEL)
        & (summary_df["wave_case"] == MAIN_WAVE_CASE)
        & (summary_df["version"] == MAIN_VERSION)
        & (summary_df["quantile_canonical"] == MAIN_QUANTILE)
        & (summary_df["scenario"].isin(SELECTED_SCENARIOS))
        & (summary_df["year"].isin(SELECTED_YEARS))
    ].copy()

    row_keys = []
    row_labels = []
    for year in SELECTED_YEARS:
        for scenario in SELECTED_SCENARIOS:
            row_keys.append((scenario, year))
            row_labels.append(f"{year} | {scenario_label(scenario)}")

    col_ids = [1, 2, 3]
    col_labels = [WETLAND_STYLE[c]["pretty"] for c in col_ids]

    mat_mean = np.full((len(row_keys), len(col_ids)), np.nan)
    mat_frac = np.full((len(row_keys), len(col_ids)), np.nan)
    mat_sev = np.full((len(row_keys), len(col_ids)), np.nan)

    for i, (scenario, year) in enumerate(row_keys):
        for j, class_id in enumerate(col_ids):
            sel = sub[
                (sub["scenario"] == scenario)
                & (sub["year"] == year)
                & (sub["wetland_class_id"] == class_id)
            ]
            if len(sel) == 0:
                continue

            mat_mean[i, j] = float(sel["delta_mean"].iloc[0])
            mat_frac[i, j] = 100.0 * float(sel["frac_linked_to_gt25_decrease"].iloc[0])
            # Fractional-loss conversion cancels in this relative metric.
            mat_sev[i, j] = float(sel["severity_rel_to_main_pct"].iloc[0])

    return mat_mean, mat_frac, mat_sev, row_labels, col_labels


def draw_heatmap_with_horizontal_cbar(
    ax,
    cax,
    mat: np.ndarray,
    row_labels: List[str],
    col_labels: List[str],
    cmap,
    vmin: Optional[float],
    vmax: Optional[float],
    cbar_label: str,
    show_yaxis: bool = True,
    value_format: str = "{:.1f}",
) -> None:
    """
    Draw one heatmap and place a horizontal colorbar directly below it.

    Parameters
    ----------
    show_yaxis : bool
        If True, display y-axis ticks and labels.
        If False, hide y-axis ticks and labels while keeping the same matrix geometry.
    """
    im = ax.imshow(mat, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)

    ax.set_xticks(np.arange(len(col_labels)))
    ax.set_xticklabels(col_labels)

    ax.set_yticks(np.arange(len(row_labels)))
    if show_yaxis:
        ax.set_yticklabels(row_labels)
    else:
        ax.set_yticklabels([])
        ax.tick_params(axis="y", left=False, labelleft=False)

    if vmin is not None and vmax is not None:
        threshold = 0.5 * (vmin + vmax)
    else:
        finite_values = mat[np.isfinite(mat)]
        threshold = float(np.nanmedian(finite_values)) if finite_values.size > 0 else None

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            if np.isfinite(val):
                text_color = "white" if (threshold is not None and val > threshold) else "black"
                ax.text(
                    j,
                    i,
                    value_format.format(val),
                    ha="center",
                    va="center",
                    fontsize=TINY_FONT,
                    color=text_color,
                )

    cbar = plt.colorbar(im, cax=cax, orientation="horizontal")
    cbar.set_label(cbar_label)


# =============================================================================
# Figure builders
# =============================================================================

def make_scenario_year_tau_severity_1x3_figure(summary_df: pd.DataFrame) -> None:
    """
    Create a new 1x3 figure:
    a) mean tauwave change
    b) wetlands linked to >25% tauwave decrease
    c) stress-weighted wetland area relative to main case

    Only the left panel shows the y-axis labels because all three panels share
    the same row categories.
    """
    mat_mean, mat_frac, mat_sev, row_labels, col_labels = build_scenario_year_matrices(summary_df)

    frac_vmax = max(5.0, safe_nanmax(mat_frac, fallback=5.0))
    sev_vmax = max(100.0, safe_nanmax(mat_sev, fallback=100.0))

    fig = plt.figure(figsize=SCENARIO_1X3_FIGSIZE)
    gs = fig.add_gridspec(
        nrows=2,
        ncols=3,
        height_ratios=[1.0, SCENARIO_1X3_CBAR_HEIGHT_RATIO],
        wspace=SCENARIO_1X3_WSPACE,
        hspace=SCENARIO_1X3_HSPACE,
    )

    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[0, 1])
    ax_c = fig.add_subplot(gs[0, 2])

    cax_a = fig.add_subplot(gs[1, 0])
    cax_b = fig.add_subplot(gs[1, 1])
    cax_c = fig.add_subplot(gs[1, 2])

    # Panel a uses its own colormap
    draw_heatmap_with_horizontal_cbar(
        ax=ax_a,
        cax=cax_a,
        mat=mat_mean,
        row_labels=row_labels,
        col_labels=col_labels,
        cmap=PANEL_A_CMAP,
        vmin=TAU_MEAN_VMIN,
        vmax=TAU_MEAN_VMAX,
        cbar_label=r"Mean $\tau_{\mathrm{wave}}$ change (%)",
        show_yaxis=True,
    )
    annotate_panel(ax_a, "a")

    # Panels b and c share the same colormap
    draw_heatmap_with_horizontal_cbar(
        ax=ax_b,
        cax=cax_b,
        mat=mat_frac,
        row_labels=row_labels,
        col_labels=col_labels,
        cmap=COMMON_HEATMAP_CMAP,
        vmin=TAU_FRACTION_VMIN,
        vmax=frac_vmax,
        cbar_label=r"Wetlands linked to >25% $\tau_{\mathrm{wave}}$ decrease (%)",
        show_yaxis=False,
    )
    annotate_panel(ax_b, "b")

    draw_heatmap_with_horizontal_cbar(
        ax=ax_c,
        cax=cax_c,
        mat=mat_sev,
        row_labels=row_labels,
        col_labels=col_labels,
        cmap=COMMON_HEATMAP_CMAP,
        vmin=SEVERITY_VMIN,
        vmax=sev_vmax,
        cbar_label="Stress-weighted wetland area\n(relative to main case, %)",
        show_yaxis=False,
    )
    annotate_panel(ax_c, "c")

    out_path = FIGURE_DIR / "Sensitivity1_4_scenario_year_tau_severity_1x3.jpg"
    save_jpg(fig, out_path)


def make_kernel_sensitivity_tau_figure(summary_df: pd.DataFrame) -> None:
    """
    Regenerate Sensitivity3_kernel_tau.jpg with:
    - corrected tauwave formatting
    - panel a y-axis upper limit = -15
    - panel b legend moved to upper right
    """
    sub = summary_df[
        (summary_df["wave_case"] == MAIN_WAVE_CASE)
        & (summary_df["version"] == MAIN_VERSION)
        & (summary_df["scenario"] == MAIN_SCENARIO)
        & (summary_df["year"] == MAIN_YEAR)
        & (summary_df["quantile_canonical"] == MAIN_QUANTILE)
    ].copy()

    kernel_order = [k["label"] for k in KERNEL_SPECS]
    kernel_labels = [f"{int(k['lambda_km'])}/{int(k['dmax_km'])}" for k in KERNEL_SPECS]

    fig, axes = plt.subplots(1, 2, figsize=KERNEL_FIGSIZE)
    x = np.arange(len(kernel_order), dtype=float)

    for class_id in [1, 2, 3]:
        ys_mean = []
        ys_frac = []

        for kernel_label in kernel_order:
            sel = sub[
                (sub["kernel_label"] == kernel_label)
                & (sub["wetland_class_id"] == class_id)
            ]

            ys_mean.append(float(sel["delta_mean"].iloc[0]) if len(sel) > 0 else np.nan)
            ys_frac.append(
                100.0 * float(sel["frac_linked_to_gt25_decrease"].iloc[0])
                if len(sel) > 0 else np.nan
            )

        axes[0].plot(
            x,
            ys_mean,
            marker="o",
            ms=KERNEL_MARKER_SIZE,
            lw=KERNEL_LINEWIDTH,
            color=WETLAND_STYLE[class_id]["color"],
            label=WETLAND_STYLE[class_id]["pretty"],
        )

        axes[1].plot(
            x,
            ys_frac,
            marker="o",
            ms=KERNEL_MARKER_SIZE,
            lw=KERNEL_LINEWIDTH,
            color=WETLAND_STYLE[class_id]["color"],
            label=WETLAND_STYLE[class_id]["pretty"],
        )

    axes[0].axhline(0, color="0.75", lw=0.8)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(kernel_labels)
    axes[0].set_xlabel(r"$\lambda$ / $D_{\max}$ (km)")
    axes[0].set_ylabel(r"Mean $\tau_{\mathrm{wave}}$ change (%)")
    axes[0].set_ylim(top=KERNEL_PANEL_A_YMAX)
    annotate_panel(axes[0], "a", x=-0.12, y=0.95)

    axes[1].set_xticks(x)
    axes[1].set_xticklabels(kernel_labels)
    axes[1].set_xlabel(r"$\lambda$ / $D_{\max}$ (km)")
    axes[1].set_ylabel(r"Wetlands linked to >25% $\tau_{\mathrm{wave}}$ decrease (%)")
    axes[1].set_ylim(top=KERNEL_PANEL_B_YMAX)
    annotate_panel(axes[1], "b", x=-0.12, y=0.95)
    axes[1].legend(frameon=False, loc=KERNEL_LEGEND_LOC)

    out_path = FIGURE_DIR / "Sensitivity3_kernel_tau.jpg"
    save_jpg(fig, out_path)


# =============================================================================
# Main execution
# =============================================================================

def main() -> None:
    """Main entry point."""
    if not SUMMARY_CSV.exists():
        raise FileNotFoundError(f"Summary table not found: {SUMMARY_CSV}")

    summary_df = pd.read_csv(SUMMARY_CSV)

    make_scenario_year_tau_severity_1x3_figure(summary_df)
    make_kernel_sensitivity_tau_figure(summary_df)

    print("Done.")


if __name__ == "__main__":
    main()

Saved figure: /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/Sensitivity1_4_scenario_year_tau_severity_1x3.jpg
Saved figure: /projectnb/nasa-marsh/Hangjie/Louisiana/cedric/Final_figures/Sensitivity3_kernel_tau.jpg
Done.
